# Figure 4. Event-scale cluster diagnostics

This notebook draws a representative event-scale clustering figure:

- Left: occurrence and cumulative growth of all DEC clusters, highlighting the PCS precursor cluster.
- Right top: PSD probability density functions (PSDPDFs) for each cluster.
- Right bottom: top-10 representative waveform-STFT samples for each cluster.


In [ ]:
from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from obspy import read, Stream
from scipy.signal import welch, stft


PROJECT_ROOT = Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben")
NPZ_DIR = PROJECT_ROOT / r"uilt_oneday_all\02clustering\result\DEC_result\all_parement"
METRICS_XLSX = PROJECT_ROOT / r"uilt_oneday_all\02clustering\result\DEC_result\precursor_curve_metrics\all_cluster_metrics_by_sheet.xlsx"
SEGMENT_DIR = PROJECT_ROOT / r"uilt_oneday_all\01data_preprocess"
OUT_DIR = PROJECT_ROOT / r"uilt_oneday_all\uilt_paper_figure\Fig4_event_cluster_diagnostics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT = "event2"          # change here, e.g., "event8"
TOP_K = 10                # representative samples per cluster
MAX_PSD_SAMPLES = 450     # use all samples when cluster size is smaller
WINDOW_HOURS = 24

# All figure font sizes are controlled here.
# Edit only this dictionary to rescale text across the complete figure.
FONT_SIZES = {
    "base": 7.2,
    "axis_label": 7.8,
    "axis_title": 8.2,
    "tick": 6.8,
    "legend": 6.8,
    "annotation": 7.2,
    "panel_label": 15.0,
    "colorbar_label": 7.2,
    "colorbar_tick": 6.2,
}

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": FONT_SIZES["base"],
    "axes.linewidth": 0.75,
    "axes.labelsize": FONT_SIZES["axis_label"],
    "axes.titlesize": FONT_SIZES["axis_title"],
    "xtick.labelsize": FONT_SIZES["tick"],
    "ytick.labelsize": FONT_SIZES["tick"],
    "legend.fontsize": FONT_SIZES["legend"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


## Helper functions


In [ ]:
def read_xlsx_sheet(path, sheet_name):
    # Read one xlsx sheet using only the Python standard library.
    ns = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with zipfile.ZipFile(path) as zf:
        shared = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall("m:si", ns):
                shared.append("".join(t.text or "" for t in si.findall(".//m:t", ns)))

        wb = ET.fromstring(zf.read("xl/workbook.xml"))
        rels = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
        relmap = {
            r.attrib["Id"]: r.attrib["Target"]
            for r in rels
            if r.attrib["Type"].endswith("/worksheet")
        }
        sheet_path = None
        for sh in wb.find("m:sheets", ns):
            if sh.attrib["name"] == sheet_name:
                rid = sh.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]
                target = relmap[rid]
                if target.startswith("/"):
                    sheet_path = target.lstrip("/")
                elif target.startswith("xl/"):
                    sheet_path = target
                else:
                    sheet_path = "xl/" + target
                break
        if sheet_path is None:
            raise ValueError(f"Sheet not found: {sheet_name}")

        root = ET.fromstring(zf.read(sheet_path))
        rows = []
        for row in root.findall(".//m:sheetData/m:row", ns):
            values = {}
            for c in row.findall("m:c", ns):
                col = "".join(ch for ch in c.attrib["r"] if ch.isalpha())
                v = c.find("m:v", ns)
                inline = c.find("m:is", ns)
                if inline is not None:
                    values[col] = "".join(t.text or "" for t in inline.findall(".//m:t", ns))
                    continue
                if v is None:
                    values[col] = ""
                    continue
                text = v.text or ""
                if c.attrib.get("t") == "s":
                    text = shared[int(text)]
                values[col] = text
            rows.append(values)

    def col_num(col):
        n = 0
        for ch in col:
            n = n * 26 + ord(ch) - 64
        return n

    cols = sorted({c for r in rows for c in r}, key=col_num)
    table = [[r.get(c, "") for c in cols] for r in rows]
    header = table[0]
    out = []
    for row in table[1:]:
        out.append({header[i]: row[i] for i in range(len(header))})
    return out


def to_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def zscore(values):
    values = np.asarray(values, dtype=float)
    mu = np.nanmean(values)
    sd = np.nanstd(values)
    if not np.isfinite(sd) or sd == 0:
        return np.zeros_like(values)
    return (values - mu) / sd


def pcs_selection(metrics_xlsx, event_name):
    rows = [r for r in read_xlsx_sheet(metrics_xlsx, "all_metrics_wide") if r.get("event") == event_name]
    if not rows:
        raise ValueError(f"No PCS metric rows for {event_name}")

    fields = ["z_final", "rate_acceleration", "c_value", "terminal_growth_ratio"]
    vals = {f: np.array([to_float(r.get(f)) for r in rows]) for f in fields}
    pcs = zscore(vals["z_final"]) + zscore(vals["rate_acceleration"]) - zscore(vals["c_value"]) + zscore(vals["terminal_growth_ratio"])

    candidates = []
    for i, r in enumerate(rows):
        z_final = to_float(r.get("z_final"))
        c_value = to_float(r.get("c_value"))
        terminal_flag = to_float(r.get("terminal_growth_flag"), 0)
        if z_final > 2.0 and c_value < 1.0 and terminal_flag == 1:
            candidates.append((pcs[i], z_final, to_float(r.get("terminal_growth_ratio")), to_float(r.get("sample_count")), i))

    if not candidates:
        return None, rows, pcs

    best = max(candidates)
    row = rows[best[-1]]
    return int(float(row["cluster"])), rows, pcs


def segment_path(name):
    name = str(name).replace(".\\", "").replace("./", "").replace("/", "\\")
    return SEGMENT_DIR / name


## Load DEC outputs and waveforms


In [ ]:
npz_candidates = [NPZ_DIR / f"{EVENT}.npz", NPZ_DIR / f"{EVENT}_all_parement.npz"]
npz_path = next((p for p in npz_candidates if p.exists()), None)
if npz_path is None:
    raise FileNotFoundError(f"No DEC npz found for {EVENT}: {npz_candidates}")
data = np.load(npz_path, allow_pickle=True)

labels = data["y_pred_last"].astype(int)
distances = data["distances"]
x_files = data["x_file"].astype(str)
n_clusters = int(data["n_clusters"])
clusters = list(range(n_clusters))
time_h = np.linspace(-WINDOW_HOURS, 0, len(labels), endpoint=False)

pcs_cluster, metric_rows, metric_pcs = pcs_selection(METRICS_XLSX, EVENT)
print(f"{EVENT}: n_clusters={n_clusters}, PCS cluster={pcs_cluster}")

paths = [segment_path(x) for x in x_files]
missing = [p for p in paths[:10] if not p.exists()]
if missing:
    raise FileNotFoundError(f"Example missing segment file: {missing[0]}")

stream = Stream()
for path in paths:
    stream += read(str(path))
stream.merge(method=1, fill_value=0)
trace = stream[0]
fs = float(trace.stats.sampling_rate)
samples_per_min = int(round(60 * fs))
wave_all = trace.data.astype(float)


def waveform_for_index(idx):
    i0 = int(idx * samples_per_min)
    i1 = i0 + samples_per_min
    x = wave_all[i0:i1].astype(float)
    if len(x) < samples_per_min:
        x = np.pad(x, (0, samples_per_min - len(x)), mode="constant")
    x = x - np.nanmean(x)
    return x


print(f"Sampling rate: {fs:g} Hz; segments: {len(labels)}")


## Compute PSDPDF and representative samples


In [ ]:
def choose_psd_indices(cluster_id):
    idx = np.where(labels == cluster_id)[0]
    if len(idx) <= MAX_PSD_SAMPLES:
        return idx
    return idx[np.linspace(0, len(idx) - 1, MAX_PSD_SAMPLES).astype(int)]


def representative_indices(cluster_id, top_k=TOP_K):
    idx = np.where(labels == cluster_id)[0]
    order = np.argsort(distances[idx, cluster_id])
    return idx[order[:min(top_k, len(order))]]


psd_by_cluster = {}
all_psd_values = []
freq_ref = None

for cid in clusters:
    rows = []
    for idx in choose_psd_indices(cid):
        x = waveform_for_index(idx)
        nperseg = min(1024, len(x))
        noverlap = min(512, nperseg // 2)
        f, p = welch(x, fs=fs, nperseg=nperseg, noverlap=noverlap)
        keep = (f > 0.5) & (f <= 50)
        p_db = 10 * np.log10(p[keep] + 1e-30)
        rows.append(p_db)
        all_psd_values.append(p_db)
        if freq_ref is None:
            freq_ref = f[keep]
    psd_by_cluster[cid] = np.vstack(rows)

all_psd_values = np.concatenate(all_psd_values)
psd_min, psd_max = np.nanpercentile(all_psd_values, [1, 99])
psd_bins = np.linspace(psd_min, psd_max, 150)
freq_bins = np.linspace(0.5, 50, 120)


def psdpdf_density(psd_rows):
    f2 = np.tile(freq_ref, psd_rows.shape[0])
    p2 = psd_rows.ravel()
    hist, _, _ = np.histogram2d(f2, p2, bins=[freq_bins, psd_bins])
    counts = np.histogram(f2, bins=freq_bins)[0]
    density = np.divide(hist, counts[:, None], out=np.zeros_like(hist), where=counts[:, None] > 0)
    return 100 * density


density_by_cluster = {cid: psdpdf_density(psd_by_cluster[cid]) for cid in clusters}
vmax_pdf = np.nanpercentile(np.concatenate([v.ravel() for v in density_by_cluster.values()]), 99.5)
rep_by_cluster = {cid: representative_indices(cid) for cid in clusters}

print("Cluster sizes:", {cid: int((labels == cid).sum()) for cid in clusters})
print("Representative samples:", {cid: rep_by_cluster[cid].tolist() for cid in clusters})


## Quantitative cluster diagnostics for manuscript text

This cell reports event-scale values used in the Results text. It does not change the figure-generation code.


In [ ]:
# Quantitative values for the manuscript text.
# This cell only reads variables produced above and does not modify the plotting code.

bands = {
    "1-10 Hz": (1, 10),
    "10-25 Hz": (10, 25),
    "25-45 Hz": (25, 45),
    "10-45 Hz": (10, 45),
    "30-45 Hz": (30, 45),
}

cluster_diagnostics = {}
for cid in clusters:
    idx = np.where(labels == cid)[0]
    psd = psd_by_cluster[cid]
    med_spec = np.nanmedian(psd, axis=0)
    peak_mask = (freq_ref >= 1) & (freq_ref <= 45)
    peak_freq = float(freq_ref[peak_mask][np.nanargmax(med_spec[peak_mask])])
    peak_psd = float(np.nanmax(med_spec[peak_mask]))

    band_psd = {}
    for band, (lo, hi) in bands.items():
        mask = (freq_ref >= lo) & (freq_ref < hi)
        band_psd[band] = float(np.nanmedian(psd[:, mask]))

    cluster_diagnostics[cid] = {
        "N": int(len(idx)),
        "last_1h_count": int(np.sum(idx >= len(labels) - 60)),
        "last_3h_count": int(np.sum(idx >= len(labels) - 180)),
        "last_6h_count": int(np.sum(idx >= len(labels) - 360)),
        "last_1h_percent": float(100 * np.sum(idx >= len(labels) - 60) / len(idx)),
        "last_3h_percent": float(100 * np.sum(idx >= len(labels) - 180) / len(idx)),
        "last_6h_percent": float(100 * np.sum(idx >= len(labels) - 360) / len(idx)),
        "median_psd_db": band_psd,
        "median_spectrum_peak_freq_hz": peak_freq,
        "median_spectrum_peak_psd_db": peak_psd,
    }

other_clusters = [cid for cid in clusters if cid != pcs_cluster]
pcs_excess_db = {}
for band in bands:
    other_mean = np.mean([cluster_diagnostics[cid]["median_psd_db"][band] for cid in other_clusters])
    pcs_value = cluster_diagnostics[pcs_cluster]["median_psd_db"][band]
    pcs_excess_db[band] = {
        "pcs_cluster_db": float(pcs_value),
        "other_cluster_mean_db": float(other_mean),
        "pcs_minus_other_mean_db": float(pcs_value - other_mean),
    }

print(f"Event: {EVENT}; PCS cluster: C{pcs_cluster}")
print("\nTemporal concentration")
print("cluster  N   last1h n/%    last3h n/%    last6h n/%")
for cid, d in cluster_diagnostics.items():
    print(
        f"C{cid:<2} {d['N']:>4}  "
        f"{d['last_1h_count']:>3}/{d['last_1h_percent']:>5.1f}%  "
        f"{d['last_3h_count']:>3}/{d['last_3h_percent']:>5.1f}%  "
        f"{d['last_6h_count']:>3}/{d['last_6h_percent']:>5.1f}%"
    )

print("\nMedian PSD by frequency band (dB)")
print("cluster  " + "  ".join(f"{band:>10}" for band in bands))
for cid, d in cluster_diagnostics.items():
    vals = "  ".join(f"{d['median_psd_db'][band]:>10.2f}" for band in bands)
    print(f"C{cid:<2}      {vals}")

print("\nMedian-spectrum peak within 1-45 Hz")
for cid, d in cluster_diagnostics.items():
    print(f"C{cid}: {d['median_spectrum_peak_freq_hz']:.2f} Hz, {d['median_spectrum_peak_psd_db']:.2f} dB")

print("\nPCS cluster PSD excess over other-cluster mean")
for band, d in pcs_excess_db.items():
    print(
        f"{band}: C{pcs_cluster}={d['pcs_cluster_db']:.2f} dB, "
        f"others mean={d['other_cluster_mean_db']:.2f} dB, "
        f"excess={d['pcs_minus_other_mean_db']:.2f} dB"
    )

## Draw Figure 4


In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.transforms import Bbox
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import stft

# ============================================================
# Basic settings
# ============================================================
cluster_colors = plt.get_cmap("tab10")(np.linspace(0, 1, max(n_clusters, 3)))
pcs_color = "#D95F02"
N_REPRESENTATIVES = 8

FONT_SIZES = {
    "base": 9.2,
    "axis_label": 9.8,
    "axis_title": 10.2,
    "tick": 8.6,
    "legend": 8.5,
    "annotation": 9.0,
    "panel_label": 15.0,
    "colorbar_label": 8.2,
    "colorbar_tick": 8.0,
}

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": FONT_SIZES["base"],
    "axes.linewidth": 0.75,
    "axes.labelsize": FONT_SIZES["axis_label"],
    "axes.titlesize": FONT_SIZES["axis_title"],
    "xtick.labelsize": FONT_SIZES["tick"],
    "ytick.labelsize": FONT_SIZES["tick"],
    "legend.fontsize": FONT_SIZES["legend"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# ============================================================
# Figure layout
# Two columns:
#   left  = temporal evolution
#   right = PSD + waveform/STFT
#
# right=0.92 intentionally leaves space for independent
# colorbars added later using fig.add_axes().
# ============================================================
fig = plt.figure(figsize=(12.3, 8.6), dpi=320)

outer = GridSpec(
    2, 2,
    figure=fig,
    width_ratios=[0.29, 0.71],
    height_ratios=[0.20, 0.80],
    left=0.055,
    right=0.92,
    bottom=0.065,
    top=0.95,
    wspace=0.23,
    hspace=0.20,
)

# ============================================================
# a. DEC sequence + cumulative occurrence curves
# ============================================================
left = GridSpecFromSubplotSpec(
    n_clusters + 1,
    1,
    subplot_spec=outer[:, 0],
    height_ratios=[0.72] + [1] * n_clusters,
    hspace=0.18,
)

# ------------------------------------------------------------
# a1. DEC state sequence
# ------------------------------------------------------------
ax_seq = fig.add_subplot(left[0, 0])

for cid in clusters:
    idx = labels == cid
    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]
    ax_seq.scatter(
        time_h[idx],
        np.full(idx.sum(), cid),
        s=4.0 if cid == pcs_cluster else 2.1,
        color=color,
        alpha=0.95 if cid == pcs_cluster else 0.38,
        linewidths=0
    )

if pcs_cluster is not None:
    ax_seq.axhline(
        pcs_cluster,
        color=pcs_color,
        lw=0.8,
        alpha=0.45
    )

ax_seq.set_xlim(-WINDOW_HOURS, 0)
ax_seq.set_yticks(clusters)
ax_seq.set_ylabel("DEC state")
ax_seq.tick_params(labelbottom=False)
ax_seq.grid(True, color="#DDDDDD", lw=0.35, alpha=0.65)

ax_seq.text(
    -0.18, 1.18, "a",
    transform=ax_seq.transAxes,
    fontsize=FONT_SIZES["panel_label"],
    fontweight="bold",
    va="top"
)

# ------------------------------------------------------------
# a2. Cumulative occurrence for each state
#
# Left y-axis  = cumulative count
# Right y-axis = cumulative fraction (0-1)
# ------------------------------------------------------------
for i, cid in enumerate(clusters):
    ax = fig.add_subplot(left[i + 1, 0], sharex=ax_seq)

    hit = (labels == cid).astype(int)
    cum = np.cumsum(hit)
    n_state = int(hit.sum())
    n_scale = max(n_state, 1)

    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]
    line_alpha = 0.98 if cid == pcs_cluster else 0.60
    line_width = 1.45 if cid == pcs_cluster else 0.90

    # Occurrence positions
    for x in time_h[hit == 1]:
        ax.axvline(
            x,
            color=color,
            lw=0.30,
            alpha=0.20 if cid == pcs_cluster else 0.10
        )

    # Raw cumulative count
    ax.plot(
        time_h,
        cum,
        color=color,
        lw=line_width,
        alpha=line_alpha
    )

    ax.fill_between(
        time_h,
        0,
        cum,
        color=color,
        alpha=0.075 if cid == pcs_cluster else 0.025
    )

    # Left axis: cumulative count
    ax.set_ylim(0, n_scale)
    ax.set_ylabel("Cum. count", labelpad=2)
    ax.grid(True, color="#DDDDDD", lw=0.35, alpha=0.55)

    # Right axis: cumulative fraction
    ax_prob = ax.secondary_yaxis(
        "right",
        functions=(
            lambda count, n=n_scale: count / n,
            lambda fraction, n=n_scale: fraction * n
        )
    )

    ax_prob.set_yticks([0.0, 0.5, 1.0])
    ax_prob.set_yticklabels(["0", "0.5", "1"])
    ax_prob.tick_params(
        axis="y",
        labelsize=FONT_SIZES["tick"],
        pad=1.5,
        length=2.5
    )

    # Only show the right-axis title once
    if i == n_clusters // 2:
        ax_prob.set_ylabel(
            "Cumulative fraction",
            fontsize=FONT_SIZES["axis_label"],
            labelpad=3
        )

    # State label
    ax.text(
        0.015,
        0.78,
        f"C{cid}" + (" | PCS" if cid == pcs_cluster else ""),
        transform=ax.transAxes,
        color=pcs_color if cid == pcs_cluster else "#555555",
        fontsize=FONT_SIZES["annotation"],
        fontweight="bold" if cid == pcs_cluster else "normal"
    )

    # Only bottom subplot displays x tick labels
    if i < n_clusters - 1:
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel("Time before debris-flow initiation (h)")

    # Highlight PCS state
    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color)
            spine.set_linewidth(1.05)

# ============================================================
# b. PSD probability-density distributions
# ============================================================
psd_grid = GridSpecFromSubplotSpec(
    1,
    n_clusters,
    subplot_spec=outer[0, 1],
    wspace=0.10
)

psd_axes = []
pdf_mappable = None

for j, cid in enumerate(clusters):
    ax = fig.add_subplot(psd_grid[0, j])
    psd_axes.append(ax)

    den = density_by_cluster[cid]

    pdf_mappable = ax.pcolormesh(
        freq_bins,
        psd_bins,
        den.T,
        cmap="YlOrRd",
        shading="auto",
        vmin=0,
        vmax=vmax_pdf,
        rasterized=True
    )

    median_psd = np.nanmedian(psd_by_cluster[cid], axis=0)

    ax.plot(
        freq_ref,
        median_psd,
        color=pcs_color if cid == pcs_cluster else "#12664F",
        lw=1.10
    )

    ax.set_xlim(0, 50)
    ax.set_ylim(psd_min, psd_max)

    ax.set_title(
        f"C{cid}" +
        (" | PCS" if cid == pcs_cluster else "") +
        f"\nn={(labels == cid).sum()}",
        color=pcs_color if cid == pcs_cluster else "black",
        fontweight="bold" if cid == pcs_cluster else "normal",
        pad=3
    )

    ax.set_xlabel("Frequency (Hz)")
    ax.grid(True, color="#EEEEEE", lw=0.32, alpha=0.55)

    if j == 0:
        ax.set_ylabel("PSD (dB)")
        ax.text(
            -0.34, 1.18, "b",
            transform=ax.transAxes,
            fontsize=FONT_SIZES["panel_label"],
            fontweight="bold",
            va="top"
        )
    else:
        ax.tick_params(labelleft=False)

    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color)
            spine.set_linewidth(1.0)

# ============================================================
# c. First eight representative waveform + STFT samples
# ============================================================
stft_grid = GridSpecFromSubplotSpec(
    N_REPRESENTATIVES,
    n_clusters,
    subplot_spec=outer[1, 1],
    wspace=0.08,
    hspace=0.055
)

stft_axes = []
last_im = None

for j, cid in enumerate(clusters):
    reps = rep_by_cluster[cid][:N_REPRESENTATIVES]

    for r in range(N_REPRESENTATIVES):
        cell = GridSpecFromSubplotSpec(
            2,
            1,
            subplot_spec=stft_grid[r, j],
            height_ratios=[0.24, 0.76],
            hspace=0.015
        )

        ax_w = fig.add_subplot(cell[0, 0])
        ax_s = fig.add_subplot(cell[1, 0])
        stft_axes.append(ax_s)

        if r < len(reps):
            x = waveform_for_index(reps[r])
            t = np.arange(len(x)) / fs

            # Waveform normalization
            scale = np.nanpercentile(np.abs(x), 99)
            x_norm = x / scale if scale > 0 else x

            ax_w.plot(
                t,
                x_norm,
                color="#303030",
                lw=0.30
            )

            # STFT
            nperseg_use = min(512, len(x))
            noverlap_use = min(
                448,
                max(0, nperseg_use - 1)
            )

            f, tt, zz = stft(
                x,
                fs=fs,
                nperseg=nperseg_use,
                noverlap=noverlap_use
            )

            keep = f <= 50
            amp = np.abs(zz[keep])

            # Self normalization
            amp_scale = np.nanpercentile(amp, 99.5) + 1e-12
            amp = amp / amp_scale
            amp = np.clip(amp, 0, 1)

            last_im = ax_s.pcolormesh(
                tt,
                f[keep],
                amp,
                cmap="turbo",
                shading="auto",
                vmin=0,
                vmax=1,
                rasterized=True
            )

        # Axis limits
        ax_w.set_xlim(0, 60)
        ax_w.set_ylim(-1.1, 1.1)
        ax_s.set_xlim(0, 60)
        ax_s.set_ylim(0, 50)

        # General appearance
        for ax in (ax_w, ax_s):
            ax.set_xticks([])
            ax.set_yticks([])

            for spine in ax.spines.values():
                spine.set_linewidth(0.25)
                spine.set_color("#777777")

        # Highlight PCS column
        if cid == pcs_cluster:
            for ax in (ax_w, ax_s):
                for spine in ax.spines.values():
                    spine.set_color(pcs_color)
                    spine.set_linewidth(0.55)

        # Column title
#         if r == 0:
#             ax_w.set_title(
#                 f"C{cid}" + (" | PCS" if cid == pcs_cluster else ""),
#                 color=pcs_color if cid == pcs_cluster else "black",
#                 fontweight="bold" if cid == pcs_cluster else "normal",
#                 pad=4
#             )

        # Panel label
        if j == 0 and r == 0:
            ax_w.text(
                -0.34, 1.85, "c",
                transform=ax_w.transAxes,
                fontsize=FONT_SIZES["panel_label"],
                fontweight="bold",
                va="top"
            )

        # Bottom x axis
        if r == N_REPRESENTATIVES - 1:
            ax_s.set_xticks([0, 30, 60])
            ax_s.set_xlabel("Time (s)", labelpad=1)

        # Left y axis
        if j == 0:
            ax_s.set_yticks([0, 25, 50])
            ax_s.set_ylabel("Hz", labelpad=1)

# ============================================================
# Independent colorbars
#
# These are NOT part of GridSpec.
# Change XPAD / WIDTH / YSHRINK independently.
# ============================================================
fig.canvas.draw()

# ------------------------------------------------------------
# PSD colorbar
# ------------------------------------------------------------
bbox_psd = Bbox.union([ax.get_position() for ax in psd_axes])

PSD_CB_XPAD = 0.008      # smaller = closer to PSD panels
PSD_CB_WIDTH = 0.007
PSD_CB_YSHRINK = 0.000

cax_pdf = fig.add_axes([
    bbox_psd.x1 + PSD_CB_XPAD,
    bbox_psd.y0 + PSD_CB_YSHRINK,
    PSD_CB_WIDTH,
    bbox_psd.height - 2 * PSD_CB_YSHRINK
])

cb = fig.colorbar(
    pdf_mappable,
    cax=cax_pdf,
    orientation="vertical"
)

cb.set_label(
    "Probability density (%)",
    fontsize=FONT_SIZES["colorbar_label"],
    labelpad=3
)

cb.ax.tick_params(
    labelsize=FONT_SIZES["colorbar_tick"],
    pad=1.5,
    length=2.5
)

# ------------------------------------------------------------
# STFT colorbar
# ------------------------------------------------------------
bbox_stft = Bbox.union([ax.get_position() for ax in stft_axes])

STFT_CB_XPAD = 0.008     # smaller = closer to STFT panels
STFT_CB_WIDTH = 0.007
STFT_CB_YSHRINK = 0.005

cax_stft = fig.add_axes([
    bbox_stft.x1 + STFT_CB_XPAD,
    bbox_stft.y0 + STFT_CB_YSHRINK,
    STFT_CB_WIDTH,
    bbox_stft.height - 2 * STFT_CB_YSHRINK
])

cb2 = fig.colorbar(
    last_im,
    cax=cax_stft,
    orientation="vertical"
)

cb2.set_label(
    "Self-normalized STFT amplitude",
    fontsize=FONT_SIZES["colorbar_label"],
    labelpad=3
)

cb2.ax.tick_params(
    labelsize=FONT_SIZES["colorbar_tick"],
    pad=1.5,
    length=2.5
)

# ============================================================
# Save
# ============================================================
out_png = OUT_DIR / f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.png"
out_svg = OUT_DIR / f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.svg"


fig.savefig(
    out_png,
    dpi=320,
    bbox_inches="tight",
    facecolor="white"
)
fig.savefig(
    out_svg,
    bbox_inches="tight",
    facecolor="white"
)



plt.show()

print(f"PNG saved to: {out_png}")
print(f"SVG saved to: {out_svg}")

In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.transforms import Bbox
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import stft

# ========================== EDIT FIGURE STYLE HERE ==========================
# Basic settings
N_REPRESENTATIVES = 7
CLUSTER_CMAP = "tab10"
PCS_COLOR = "#D95F02"
NORMAL_COLOR = "#12664F"

# Figure size and outer layout
FIGSIZE = (12.3, 8.6)
FIGURE_DPI = 320
OUTER_WIDTH_RATIOS = (0.29, 0.71)
OUTER_HEIGHT_RATIOS = (0.20, 0.80)
OUTER_LEFT = 0.055
OUTER_RIGHT = 0.92
OUTER_BOTTOM = 0.065
OUTER_TOP = 0.95
OUTER_WSPACE = 0.18
OUTER_HSPACE = 0.20

# Internal subplot spacing
LEFT_HEIGHT_RATIOS_FIRST = 0.72
LEFT_HSPACE = 0.18
PSD_WSPACE = 0.10
STFT_WSPACE = 0.08
STFT_HSPACE = 0.055
WAVEFORM_STFT_HEIGHT_RATIOS = (0.24, 0.76)
WAVEFORM_STFT_HSPACE = 0.015
add_font = 1
# Font sizes
FONT_SIZES = {
    "base": 9.2, "axis_label": 9.8, "axis_title": 10.2, "tick": 8.6,
    "legend": 8.5, "annotation": 9.0, "panel_label": 15.0,
    "colorbar_label": 8.2, "colorbar_tick": 8.0,
}

# Panel-label positions in axes coordinates
PANEL_LABEL_POSITION = {"a": (-0.18, 1.18), "b": (-0.34, 1.18), "c": (-0.34, 1.85)}

# State-label position
STATE_LABEL_POSITION = (0.015, 0.78)

# DEC sequence appearance
SEQUENCE_PCS_SIZE = 4.0
SEQUENCE_OTHER_SIZE = 2.1
SEQUENCE_PCS_ALPHA = 0.95
SEQUENCE_OTHER_ALPHA = 0.38
PCS_HORIZONTAL_LINEWIDTH = 0.8
PCS_HORIZONTAL_ALPHA = 0.45

# Cumulative-curve appearance
CUMULATIVE_PCS_LINEWIDTH = 1.45
CUMULATIVE_OTHER_LINEWIDTH = 0.90
CUMULATIVE_PCS_ALPHA = 0.98
CUMULATIVE_OTHER_ALPHA = 0.60
OCCURRENCE_LINEWIDTH = 0.30
OCCURRENCE_PCS_ALPHA = 0.20
OCCURRENCE_OTHER_ALPHA = 0.10
CUMULATIVE_PCS_FILL_ALPHA = 0.075
CUMULATIVE_OTHER_FILL_ALPHA = 0.025
PCS_SPINE_LINEWIDTH = 1.05

# Grid appearance
GRID_COLOR = "#DDDDDD"
SEQUENCE_GRID_LINEWIDTH = 0.35
SEQUENCE_GRID_ALPHA = 0.65
CUMULATIVE_GRID_LINEWIDTH = 0.35
CUMULATIVE_GRID_ALPHA = 0.55
PSD_GRID_COLOR = "#EEEEEE"
PSD_GRID_LINEWIDTH = 0.32
PSD_GRID_ALPHA = 0.55

# PSD appearance
PSD_CMAP = "YlOrRd"
PSD_FREQUENCY_LIMIT = (0, 50)
PSD_MEDIAN_LINEWIDTH = 1.10
PSD_TITLE_PAD = 3
PSD_PCS_SPINE_LINEWIDTH = 1.0

# Waveform and STFT appearance
WAVEFORM_COLOR = "#303030"
WAVEFORM_LINEWIDTH = 0.30
WAVEFORM_TIME_LIMIT = (0, 60)
WAVEFORM_AMPLITUDE_LIMIT = (-1.1, 1.1)
STFT_FREQUENCY_LIMIT = (0, 50)
WAVEFORM_NORMALIZATION_PERCENTILE = 99
STFT_NORMALIZATION_PERCENTILE = 99.5
STFT_NPERSEG = 512
STFT_NOVERLAP = 448
STFT_CMAP = "turbo"
STFT_VMIN = 0
STFT_VMAX = 1
NORMAL_SPINE_COLOR = "#777777"
NORMAL_SPINE_LINEWIDTH = 0.25
PCS_STFT_SPINE_LINEWIDTH = 0.55

# Tick and label spacing
CUMULATIVE_COUNT_LABEL_PAD = 2
CUMULATIVE_FRACTION_LABEL_PAD = 3
SECONDARY_AXIS_TICK_PAD = 1.5
SECONDARY_AXIS_TICK_LENGTH = 2.5
BOTTOM_TIME_LABEL_PAD = 1
LEFT_FREQUENCY_LABEL_PAD = 1

# Independent colorbar positions
PSD_CB_XPAD = 0.008
PSD_CB_WIDTH = 0.007
PSD_CB_YSHRINK = 0.000
STFT_CB_XPAD = 0.008
STFT_CB_WIDTH = 0.007
STFT_CB_YSHRINK = 0.005
COLORBAR_LABEL_PAD = 3
COLORBAR_TICK_PAD = 1.5
COLORBAR_TICK_LENGTH = 2.5

# Export settings
SAVE_DPI = 320
PNG_NAME = f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.png"
SVG_NAME = f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.tiff"
# ===========================================================================

cluster_colors = plt.get_cmap(CLUSTER_CMAP)(np.linspace(0, 1, max(n_clusters, 3)))
pcs_color = PCS_COLOR

mpl.rcParams.update({
    "font.family": "Arial", "font.size": FONT_SIZES["base"], "axes.linewidth": 0.75,
    "axes.labelsize": FONT_SIZES["axis_label"], "axes.titlesize": FONT_SIZES["axis_title"],
    "xtick.labelsize": FONT_SIZES["tick"], "ytick.labelsize": FONT_SIZES["tick"],
    "legend.fontsize": FONT_SIZES["legend"], "pdf.fonttype": 42,
    "ps.fonttype": 42, "svg.fonttype": "none",
})

fig = plt.figure(figsize=FIGSIZE, dpi=FIGURE_DPI)
outer = GridSpec(
    2, 2, figure=fig, width_ratios=OUTER_WIDTH_RATIOS, height_ratios=OUTER_HEIGHT_RATIOS,
    left=OUTER_LEFT, right=OUTER_RIGHT, bottom=OUTER_BOTTOM, top=OUTER_TOP,
    wspace=OUTER_WSPACE, hspace=OUTER_HSPACE,
)

# a. DEC sequence and cumulative occurrence curves
left = GridSpecFromSubplotSpec(
    n_clusters + 1, 1, subplot_spec=outer[:, 0],
    height_ratios=[LEFT_HEIGHT_RATIOS_FIRST] + [1] * n_clusters, hspace=LEFT_HSPACE,
)

# a1. DEC state sequence
ax_seq = fig.add_subplot(left[0, 0])
for cid in clusters:
    idx = labels == cid
    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]
    ax_seq.scatter(
        time_h[idx], np.full(idx.sum(), cid),
        s=SEQUENCE_PCS_SIZE if cid == pcs_cluster else SEQUENCE_OTHER_SIZE,
        color=color, alpha=SEQUENCE_PCS_ALPHA if cid == pcs_cluster else SEQUENCE_OTHER_ALPHA,
        linewidths=0,
    )

if pcs_cluster is not None:
    ax_seq.axhline(
        pcs_cluster, color=pcs_color,
        lw=PCS_HORIZONTAL_LINEWIDTH, alpha=PCS_HORIZONTAL_ALPHA,
    )

ax_seq.set_xlim(-WINDOW_HOURS, 0)
ax_seq.set_yticks(clusters)
ax_seq.set_ylabel("DEC state")
ax_seq.tick_params(labelbottom=False)
ax_seq.grid(True, color=GRID_COLOR, lw=SEQUENCE_GRID_LINEWIDTH, alpha=SEQUENCE_GRID_ALPHA)
ax_seq.text(
    *PANEL_LABEL_POSITION["a"], "a", transform=ax_seq.transAxes,
    fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top",
)

# a2. Cumulative occurrence for each state
for i, cid in enumerate(clusters):
    ax = fig.add_subplot(left[i + 1, 0], sharex=ax_seq)
    hit = (labels == cid).astype(int)
    cum = np.cumsum(hit)
    n_state = int(hit.sum())
    n_scale = max(n_state, 1)
    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]
    line_alpha = CUMULATIVE_PCS_ALPHA if cid == pcs_cluster else CUMULATIVE_OTHER_ALPHA
    line_width = CUMULATIVE_PCS_LINEWIDTH if cid == pcs_cluster else CUMULATIVE_OTHER_LINEWIDTH

    for x in time_h[hit == 1]:
        ax.axvline(
            x, color=color, lw=OCCURRENCE_LINEWIDTH,
            alpha=OCCURRENCE_PCS_ALPHA if cid == pcs_cluster else OCCURRENCE_OTHER_ALPHA,
        )

    ax.plot(time_h, cum, color=color, lw=line_width, alpha=line_alpha)
    ax.fill_between(
        time_h, 0, cum, color=color,
        alpha=CUMULATIVE_PCS_FILL_ALPHA if cid == pcs_cluster else CUMULATIVE_OTHER_FILL_ALPHA,
    )
    ax.set_ylim(0, n_scale)
    ax.set_ylabel("Cum. count", labelpad=CUMULATIVE_COUNT_LABEL_PAD)
    ax.grid(True, color=GRID_COLOR, lw=CUMULATIVE_GRID_LINEWIDTH, alpha=CUMULATIVE_GRID_ALPHA)

    ax_prob = ax.secondary_yaxis(
        "right",
        functions=(lambda count, n=n_scale: count / n, lambda fraction, n=n_scale: fraction * n),
    )
    ax_prob.set_yticks([0.0, 0.5, 1.0])
    ax_prob.set_yticklabels(["0", "0.5", "1"])
    ax_prob.tick_params(
        axis="y", labelsize=FONT_SIZES["tick"],
        pad=SECONDARY_AXIS_TICK_PAD, length=SECONDARY_AXIS_TICK_LENGTH,
    )

    if i == n_clusters // 2:
        ax_prob.set_ylabel(
            "Probability", fontsize=FONT_SIZES["axis_label"],
            labelpad=CUMULATIVE_FRACTION_LABEL_PAD,
        )

    ax.text(
        *STATE_LABEL_POSITION, f"C{cid}" + (" | PCS" if cid == pcs_cluster else ""),
        transform=ax.transAxes, color=pcs_color if cid == pcs_cluster else "#555555",
        fontsize=FONT_SIZES["annotation"],
        fontweight="bold" if cid == pcs_cluster else "normal",
    )

    if i < n_clusters - 1:
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel("Time before debris-flow initiation (h)")

    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color)
            spine.set_linewidth(PCS_SPINE_LINEWIDTH)

# b. PSD probability-density distributions
psd_grid = GridSpecFromSubplotSpec(1, n_clusters, subplot_spec=outer[0, 1], wspace=PSD_WSPACE)
psd_axes = []
pdf_mappable = None

for j, cid in enumerate(clusters):
    ax = fig.add_subplot(psd_grid[0, j])
    psd_axes.append(ax)
    den = density_by_cluster[cid]
    pdf_mappable = ax.pcolormesh(
        freq_bins, psd_bins, den.T, cmap=PSD_CMAP, shading="auto",
        vmin=0, vmax=vmax_pdf, rasterized=True,
    )
    median_psd = np.nanmedian(psd_by_cluster[cid], axis=0)
    ax.plot(
        freq_ref, median_psd,
        color=pcs_color if cid == pcs_cluster else NORMAL_COLOR,
        lw=PSD_MEDIAN_LINEWIDTH,
    )
    ax.set_xlim(*PSD_FREQUENCY_LIMIT)
    ax.set_ylim(psd_min, psd_max)
    ax.set_title(
        f"C{cid}" + (" | PCS" if cid == pcs_cluster else "") + f"\nn={(labels == cid).sum()}",
        color=pcs_color if cid == pcs_cluster else "black",
        fontweight="bold" if cid == pcs_cluster else "normal", pad=PSD_TITLE_PAD,
    )
    ax.set_xlabel("Frequency (Hz)")
    ax.grid(True, color=PSD_GRID_COLOR, lw=PSD_GRID_LINEWIDTH, alpha=PSD_GRID_ALPHA)

    if j == 0:
        ax.set_ylabel("PSD (dB)")
        ax.text(
            *PANEL_LABEL_POSITION["b"], "b", transform=ax.transAxes,
            fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top",
        )
    else:
        ax.tick_params(labelleft=False)

    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color)
            spine.set_linewidth(PSD_PCS_SPINE_LINEWIDTH)

# c. First eight representative waveform and STFT samples
stft_grid = GridSpecFromSubplotSpec(
    N_REPRESENTATIVES, n_clusters, subplot_spec=outer[1, 1],
    wspace=STFT_WSPACE, hspace=STFT_HSPACE,
)
stft_axes = []
last_im = None

for j, cid in enumerate(clusters):
    reps = rep_by_cluster[cid][:N_REPRESENTATIVES]
    for r in range(N_REPRESENTATIVES):
        cell = GridSpecFromSubplotSpec(
            2, 1, subplot_spec=stft_grid[r, j],
            height_ratios=WAVEFORM_STFT_HEIGHT_RATIOS, hspace=WAVEFORM_STFT_HSPACE,
        )
        ax_w = fig.add_subplot(cell[0, 0])
        ax_s = fig.add_subplot(cell[1, 0])
        stft_axes.append(ax_s)

        if r < len(reps):
            x = waveform_for_index(reps[r])
            t = np.arange(len(x)) / fs
            scale = np.nanpercentile(np.abs(x), WAVEFORM_NORMALIZATION_PERCENTILE)
            x_norm = x / scale if scale > 0 else x
            ax_w.plot(t, x_norm, color=WAVEFORM_COLOR, lw=WAVEFORM_LINEWIDTH)

            nperseg_use = min(STFT_NPERSEG, len(x))
            noverlap_use = min(STFT_NOVERLAP, max(0, nperseg_use - 1))
            f, tt, zz = stft(x, fs=fs, nperseg=nperseg_use, noverlap=noverlap_use)
            keep = f <= STFT_FREQUENCY_LIMIT[1]
            amp = np.abs(zz[keep])
            amp_scale = np.nanpercentile(amp, STFT_NORMALIZATION_PERCENTILE) + 1e-12
            amp = np.clip(amp / amp_scale, STFT_VMIN, STFT_VMAX)
            last_im = ax_s.pcolormesh(
                tt, f[keep], amp, cmap=STFT_CMAP, shading="auto",
                vmin=STFT_VMIN, vmax=STFT_VMAX, rasterized=True,
            )

        ax_w.set_xlim(*WAVEFORM_TIME_LIMIT)
        ax_w.set_ylim(*WAVEFORM_AMPLITUDE_LIMIT)
        ax_s.set_xlim(*WAVEFORM_TIME_LIMIT)
        ax_s.set_ylim(*STFT_FREQUENCY_LIMIT)

        for ax in (ax_w, ax_s):
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(NORMAL_SPINE_LINEWIDTH)
                spine.set_color(NORMAL_SPINE_COLOR)

        if cid == pcs_cluster:
            for ax in (ax_w, ax_s):
                for spine in ax.spines.values():
                    spine.set_color(pcs_color)
                    spine.set_linewidth(PCS_STFT_SPINE_LINEWIDTH)

        # Column title
        # if r == 0:
        #     ax_w.set_title(
        #         f"C{cid}" + (" | PCS" if cid == pcs_cluster else ""),
        #         color=pcs_color if cid == pcs_cluster else "black",
        #         fontweight="bold" if cid == pcs_cluster else "normal", pad=4,
        #     )

        if j == 0 and r == 0:
            ax_w.text(
                *PANEL_LABEL_POSITION["c"], "c", transform=ax_w.transAxes,
                fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top",
            )

        if r == N_REPRESENTATIVES - 1:
            ax_s.set_xticks([0, 30, 60])
            ax_s.set_xlabel("Time (s)", labelpad=BOTTOM_TIME_LABEL_PAD)

        if j == 0:
            ax_s.set_yticks([0, 25, 50])
            ax_s.set_ylabel("Hz", labelpad=LEFT_FREQUENCY_LABEL_PAD)

# Independent colorbars
fig.canvas.draw()

bbox_psd = Bbox.union([ax.get_position() for ax in psd_axes])
cax_pdf = fig.add_axes([
    bbox_psd.x1 + PSD_CB_XPAD,
    bbox_psd.y0 + PSD_CB_YSHRINK,
    PSD_CB_WIDTH,
    bbox_psd.height - 2 * PSD_CB_YSHRINK,
])
cb = fig.colorbar(pdf_mappable, cax=cax_pdf, orientation="vertical")
cb.set_label(
    "Probability density (%)",
    fontsize=FONT_SIZES["colorbar_label"],
    labelpad=COLORBAR_LABEL_PAD,
)
cb.ax.tick_params(
    labelsize=FONT_SIZES["colorbar_tick"],
    pad=COLORBAR_TICK_PAD,
    length=COLORBAR_TICK_LENGTH,
)

bbox_stft = Bbox.union([ax.get_position() for ax in stft_axes])
cax_stft = fig.add_axes([
    bbox_stft.x1 + STFT_CB_XPAD,
    bbox_stft.y0 + STFT_CB_YSHRINK,
    STFT_CB_WIDTH,
    bbox_stft.height - 2 * STFT_CB_YSHRINK,
])
cb2 = fig.colorbar(last_im, cax=cax_stft, orientation="vertical")
cb2.set_label(
    "Normalized amplitude",
    fontsize=FONT_SIZES["colorbar_label"],
    labelpad=COLORBAR_LABEL_PAD,
)
cb2.ax.tick_params(
    labelsize=FONT_SIZES["colorbar_tick"],
    pad=COLORBAR_TICK_PAD,
    length=COLORBAR_TICK_LENGTH,
)

# Save
out_png = OUT_DIR / PNG_NAME
out_svg = OUT_DIR / SVG_NAME
fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight", facecolor="white")
fig.savefig(out_svg, bbox_inches="tight", facecolor="white")
plt.show()
print(f"PNG saved to: {out_png}")

In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.transforms import Bbox
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import stft

# ========================== EDIT FIGURE STYLE HERE ==========================
N_REPRESENTATIVES = 7
CLUSTER_CMAP = "tab10"
PCS_COLOR = "#D95F02"
NORMAL_COLOR = "#12664F"
FIGSIZE = (12.3, 8.6);FIGURE_DPI = 320;OUTER_WIDTH_RATIOS = (0.29, 0.71);OUTER_HEIGHT_RATIOS = (0.20, 0.80)
OUTER_LEFT = 0.055
OUTER_RIGHT = 0.92
OUTER_BOTTOM = 0.065
OUTER_TOP = 0.95
OUTER_WSPACE = 0.18
OUTER_HSPACE = 0.20
LEFT_HEIGHT_RATIOS_FIRST = 0.72
LEFT_HSPACE = 0.18
PSD_WSPACE = 0.10
STFT_WSPACE = 0.08
STFT_HSPACE = 0.055
WAVEFORM_STFT_HEIGHT_RATIOS = (0.24, 0.76)
WAVEFORM_STFT_HSPACE = 0.015
add_font = 2
FONT_SIZES = {"base": 9.2 + add_font, "axis_label": 9.8+ add_font, "axis_title": 10.2, "tick": 8.6, "legend": 8.5, "annotation": 9.0, "panel_label": 15.0+ add_font, "colorbar_label": 8.2, "colorbar_tick": 8.0}
PANEL_LABEL_POSITION = {"a": (-0.18, 1.18), "b": (-0.34, 1.18), "c": (-0.34, 1.85)}
STATE_LABEL_POSITION = (0.015, 0.78)
SEQUENCE_PCS_SIZE = 4.0
SEQUENCE_OTHER_SIZE = 2.1
SEQUENCE_PCS_ALPHA = 0.95
SEQUENCE_OTHER_ALPHA = 0.38
PCS_HORIZONTAL_LINEWIDTH = 0.8
PCS_HORIZONTAL_ALPHA = 0.45
CUMULATIVE_PCS_LINEWIDTH = 1.45
CUMULATIVE_OTHER_LINEWIDTH = 0.90
CUMULATIVE_PCS_ALPHA = 0.98
CUMULATIVE_OTHER_ALPHA = 0.60
OCCURRENCE_LINEWIDTH = 0.30
OCCURRENCE_PCS_ALPHA = 0.20
OCCURRENCE_OTHER_ALPHA = 0.10
CUMULATIVE_PCS_FILL_ALPHA = 0.075
CUMULATIVE_OTHER_FILL_ALPHA = 0.025
PCS_SPINE_LINEWIDTH = 1.05
GRID_COLOR = "#DDDDDD"
SEQUENCE_GRID_LINEWIDTH = 0.35
SEQUENCE_GRID_ALPHA = 0.65
CUMULATIVE_GRID_LINEWIDTH = 0.35
CUMULATIVE_GRID_ALPHA = 0.55
PSD_GRID_COLOR = "#EEEEEE"
PSD_GRID_LINEWIDTH = 0.32
PSD_GRID_ALPHA = 0.55
PSD_CMAP = "YlOrRd"
PSD_FREQUENCY_LIMIT = (0, 50)
PSD_MEDIAN_LINEWIDTH = 1.10
PSD_TITLE_PAD = 3
PSD_PCS_SPINE_LINEWIDTH = 1.0
WAVEFORM_COLOR = "#303030"
WAVEFORM_LINEWIDTH = 0.30
WAVEFORM_TIME_LIMIT = (0, 60)
WAVEFORM_AMPLITUDE_LIMIT = (-1.1, 1.1)
STFT_FREQUENCY_LIMIT = (0, 50)
WAVEFORM_NORMALIZATION_PERCENTILE = 99
STFT_NORMALIZATION_PERCENTILE = 99.5
STFT_NPERSEG = 512
STFT_NOVERLAP = 448
STFT_CMAP = "turbo"
STFT_VMIN = 0
STFT_VMAX = 1
NORMAL_SPINE_COLOR = "#777777"
NORMAL_SPINE_LINEWIDTH = 0.25
PCS_STFT_SPINE_LINEWIDTH = 0.55
CUMULATIVE_COUNT_LABEL_PAD = 2
CUMULATIVE_FRACTION_LABEL_PAD = 3
SECONDARY_AXIS_TICK_PAD = 1.5
SECONDARY_AXIS_TICK_LENGTH = 2.5
BOTTOM_TIME_LABEL_PAD = 1
LEFT_FREQUENCY_LABEL_PAD = 1
PSD_CB_XPAD = 0.008
PSD_CB_WIDTH = 0.007
PSD_CB_YSHRINK = 0.000
STFT_CB_XPAD = 0.008
STFT_CB_WIDTH = 0.007
STFT_CB_YSHRINK = 0.005
COLORBAR_LABEL_PAD = 3
COLORBAR_TICK_PAD = 1.5
COLORBAR_TICK_LENGTH = 2.5
SAVE_DPI = 320
PNG_NAME = f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.png"
SVG_NAME = f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.tiff"
# ===========================================================================

cluster_colors = plt.get_cmap(CLUSTER_CMAP)(np.linspace(0, 1, max(n_clusters, 3)))
pcs_color = PCS_COLOR
mpl.rcParams.update({"font.family": "Arial", "font.size": FONT_SIZES["base"], "axes.linewidth": 0.75, "axes.labelsize": FONT_SIZES["axis_label"], "axes.titlesize": FONT_SIZES["axis_title"], "xtick.labelsize": FONT_SIZES["tick"], "ytick.labelsize": FONT_SIZES["tick"], "legend.fontsize": FONT_SIZES["legend"], "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none"})

fig = plt.figure(figsize=FIGSIZE, dpi=FIGURE_DPI)
outer = GridSpec(2, 2, figure=fig, width_ratios=OUTER_WIDTH_RATIOS, height_ratios=OUTER_HEIGHT_RATIOS, left=OUTER_LEFT, right=OUTER_RIGHT, bottom=OUTER_BOTTOM, top=OUTER_TOP, wspace=OUTER_WSPACE, hspace=OUTER_HSPACE)

# a. DEC sequence and cumulative occurrence curves
left = GridSpecFromSubplotSpec(n_clusters + 1, 1, subplot_spec=outer[:, 0], height_ratios=[LEFT_HEIGHT_RATIOS_FIRST] + [1] * n_clusters, hspace=LEFT_HSPACE)
ax_seq = fig.add_subplot(left[0, 0])

for cid in clusters:
    idx = labels == cid
    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]
    ax_seq.scatter(time_h[idx], np.full(idx.sum(), cid), s=SEQUENCE_PCS_SIZE if cid == pcs_cluster else SEQUENCE_OTHER_SIZE, color=color, alpha=SEQUENCE_PCS_ALPHA if cid == pcs_cluster else SEQUENCE_OTHER_ALPHA, linewidths=0)

if pcs_cluster is not None:
    ax_seq.axhline(pcs_cluster, color=pcs_color, lw=PCS_HORIZONTAL_LINEWIDTH, alpha=PCS_HORIZONTAL_ALPHA)

ax_seq.set_xlim(-WINDOW_HOURS, 0)
ax_seq.set_yticks(clusters)
ax_seq.set_ylabel("DEC state")
ax_seq.tick_params(labelbottom=False)
ax_seq.grid(True, color=GRID_COLOR, lw=SEQUENCE_GRID_LINEWIDTH, alpha=SEQUENCE_GRID_ALPHA)
ax_seq.text(*PANEL_LABEL_POSITION["a"], "a", transform=ax_seq.transAxes, fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top")

for i, cid in enumerate(clusters):
    ax = fig.add_subplot(left[i + 1, 0], sharex=ax_seq)
    hit = (labels == cid).astype(int)
    cum = np.cumsum(hit)
    n_state = int(hit.sum())
    n_scale = max(n_state, 1)
    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]
    line_alpha = CUMULATIVE_PCS_ALPHA if cid == pcs_cluster else CUMULATIVE_OTHER_ALPHA
    line_width = CUMULATIVE_PCS_LINEWIDTH if cid == pcs_cluster else CUMULATIVE_OTHER_LINEWIDTH
    for x in time_h[hit == 1]:
        ax.axvline(x, color=color, lw=OCCURRENCE_LINEWIDTH, alpha=OCCURRENCE_PCS_ALPHA if cid == pcs_cluster else OCCURRENCE_OTHER_ALPHA)
    ax.plot(time_h, cum, color=color, lw=line_width, alpha=line_alpha)
    ax.fill_between(time_h, 0, cum, color=color, alpha=CUMULATIVE_PCS_FILL_ALPHA if cid == pcs_cluster else CUMULATIVE_OTHER_FILL_ALPHA)
    ax.set_ylim(0, n_scale)
    ax.set_ylabel("Cum. count", labelpad=CUMULATIVE_COUNT_LABEL_PAD)
    ax.grid(True, color=GRID_COLOR, lw=CUMULATIVE_GRID_LINEWIDTH, alpha=CUMULATIVE_GRID_ALPHA)
    ax_prob = ax.secondary_yaxis("right", functions=(lambda count, n=n_scale: count / n, lambda fraction, n=n_scale: fraction * n))
    ax_prob.set_yticks([0.0, 0.5, 1.0])
    ax_prob.set_yticklabels(["0", "0.5", "1"])
    ax_prob.tick_params(axis="y", labelsize=FONT_SIZES["tick"], pad=SECONDARY_AXIS_TICK_PAD, length=SECONDARY_AXIS_TICK_LENGTH)
    if i == n_clusters // 2:
        ax_prob.set_ylabel("Probability", fontsize=FONT_SIZES["axis_label"], labelpad=CUMULATIVE_FRACTION_LABEL_PAD)
    ax.text(*STATE_LABEL_POSITION, f"C{cid}" + (" | PCS" if cid == pcs_cluster else ""), transform=ax.transAxes, color=pcs_color if cid == pcs_cluster else "#555555", fontsize=FONT_SIZES["annotation"], fontweight="bold" if cid == pcs_cluster else "normal")
    if i < n_clusters - 1:
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel("Time before debris-flow initiation (h)")
    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color)
            spine.set_linewidth(PCS_SPINE_LINEWIDTH)

# b. PSD probability-density distributions
psd_grid = GridSpecFromSubplotSpec(1, n_clusters, subplot_spec=outer[0, 1], wspace=PSD_WSPACE)
psd_axes = []
pdf_mappable = None

for j, cid in enumerate(clusters):
    ax = fig.add_subplot(psd_grid[0, j])
    psd_axes.append(ax)
    den = density_by_cluster[cid]
    pdf_mappable = ax.pcolormesh(freq_bins, psd_bins, den.T, cmap=PSD_CMAP, shading="auto", vmin=0, vmax=vmax_pdf, rasterized=True)
    median_psd = np.nanmedian(psd_by_cluster[cid], axis=0)
    ax.plot(freq_ref, median_psd, color=pcs_color if cid == pcs_cluster else NORMAL_COLOR, lw=PSD_MEDIAN_LINEWIDTH)
    ax.set_xlim(*PSD_FREQUENCY_LIMIT)
    ax.set_ylim(psd_min, psd_max)
    ax.set_title(f"C{cid}" + (" | PCS" if cid == pcs_cluster else "") + f"\nn={(labels == cid).sum()}", color=pcs_color if cid == pcs_cluster else "black", fontweight="bold" if cid == pcs_cluster else "normal", pad=PSD_TITLE_PAD)
    ax.set_xlabel("Frequency (Hz)")
    ax.grid(True, color=PSD_GRID_COLOR, lw=PSD_GRID_LINEWIDTH, alpha=PSD_GRID_ALPHA)
    if j == 0:
        ax.set_ylabel("PSD (dB)")
        ax.text(*PANEL_LABEL_POSITION["b"], "b", transform=ax.transAxes, fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top")
    else:
        ax.tick_params(labelleft=False)
    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color)
            spine.set_linewidth(PSD_PCS_SPINE_LINEWIDTH)

# c. First eight representative waveform and STFT samples
stft_grid = GridSpecFromSubplotSpec(N_REPRESENTATIVES, n_clusters, subplot_spec=outer[1, 1], wspace=STFT_WSPACE, hspace=STFT_HSPACE)
stft_axes = []
last_im = None

for j, cid in enumerate(clusters):
    reps = rep_by_cluster[cid][:N_REPRESENTATIVES]
    for r in range(N_REPRESENTATIVES):
        cell = GridSpecFromSubplotSpec(2, 1, subplot_spec=stft_grid[r, j], height_ratios=WAVEFORM_STFT_HEIGHT_RATIOS, hspace=WAVEFORM_STFT_HSPACE)
        ax_w = fig.add_subplot(cell[0, 0])
        ax_s = fig.add_subplot(cell[1, 0])
        stft_axes.append(ax_s)
        if r < len(reps):
            x = waveform_for_index(reps[r])
            t = np.arange(len(x)) / fs
            scale = np.nanpercentile(np.abs(x), WAVEFORM_NORMALIZATION_PERCENTILE)
            x_norm = x / scale if scale > 0 else x
            ax_w.plot(t, x_norm, color=WAVEFORM_COLOR, lw=WAVEFORM_LINEWIDTH)
            nperseg_use = min(STFT_NPERSEG, len(x))
            noverlap_use = min(STFT_NOVERLAP, max(0, nperseg_use - 1))
            f, tt, zz = stft(x, fs=fs, nperseg=nperseg_use, noverlap=noverlap_use)
            keep = f <= STFT_FREQUENCY_LIMIT[1]
            amp = np.abs(zz[keep])
            amp_scale = np.nanpercentile(amp, STFT_NORMALIZATION_PERCENTILE) + 1e-12
            amp = np.clip(amp / amp_scale, STFT_VMIN, STFT_VMAX)
            last_im = ax_s.pcolormesh(tt, f[keep], amp, cmap=STFT_CMAP, shading="auto", vmin=STFT_VMIN, vmax=STFT_VMAX, rasterized=True)
        ax_w.set_xlim(*WAVEFORM_TIME_LIMIT)
        ax_w.set_ylim(*WAVEFORM_AMPLITUDE_LIMIT)
        ax_s.set_xlim(*WAVEFORM_TIME_LIMIT)
        ax_s.set_ylim(*STFT_FREQUENCY_LIMIT)
        for ax in (ax_w, ax_s):
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(NORMAL_SPINE_LINEWIDTH)
                spine.set_color(NORMAL_SPINE_COLOR)
        if cid == pcs_cluster:
            for ax in (ax_w, ax_s):
                for spine in ax.spines.values():
                    spine.set_color(pcs_color)
                    spine.set_linewidth(PCS_STFT_SPINE_LINEWIDTH)
        
        if j == 0 and r == 0:
            ax_w.text(*PANEL_LABEL_POSITION["c"], "c", transform=ax_w.transAxes, fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top")
        if r == N_REPRESENTATIVES - 1:
            ax_s.set_xticks([0, 30, 60])
            ax_s.set_xlabel("Time (s)", labelpad=BOTTOM_TIME_LABEL_PAD)
        if j == 0:
            ax_s.set_yticks([0, 25, 50])
            ax_s.set_ylabel("Freq.(Hz)", labelpad=LEFT_FREQUENCY_LABEL_PAD)

# Independent colorbars
fig.canvas.draw()
bbox_psd = Bbox.union([ax.get_position() for ax in psd_axes])
cax_pdf = fig.add_axes([bbox_psd.x1 + PSD_CB_XPAD, bbox_psd.y0 + PSD_CB_YSHRINK, PSD_CB_WIDTH, bbox_psd.height - 2 * PSD_CB_YSHRINK])
cb = fig.colorbar(pdf_mappable, cax=cax_pdf, orientation="vertical")
cb.set_label("Probability density (%)", fontsize=FONT_SIZES["colorbar_label"], labelpad=COLORBAR_LABEL_PAD)
cb.ax.tick_params(labelsize=FONT_SIZES["colorbar_tick"], pad=COLORBAR_TICK_PAD, length=COLORBAR_TICK_LENGTH)

bbox_stft = Bbox.union([ax.get_position() for ax in stft_axes])
cax_stft = fig.add_axes([bbox_stft.x1 + STFT_CB_XPAD, bbox_stft.y0 + STFT_CB_YSHRINK, STFT_CB_WIDTH, bbox_stft.height - 2 * STFT_CB_YSHRINK])
cb2 = fig.colorbar(last_im, cax=cax_stft, orientation="vertical")
cb2.set_label("Normalized amplitude", fontsize=FONT_SIZES["colorbar_label"], labelpad=COLORBAR_LABEL_PAD)
cb2.ax.tick_params(labelsize=FONT_SIZES["colorbar_tick"], pad=COLORBAR_TICK_PAD, length=COLORBAR_TICK_LENGTH)

# Save
out_png = OUT_DIR / PNG_NAME
out_svg = OUT_DIR / SVG_NAME
fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight", facecolor="white")
fig.savefig(out_svg, bbox_inches="tight", facecolor="white")
plt.show()
print(f"PNG saved to: {out_png}")

In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.transforms import Bbox
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import stft


# ========================== EDIT FIGURE STYLE HERE ==========================

N_REPRESENTATIVES = 7; CLUSTER_CMAP = "tab10"
PCS_COLOR = "#D95F02"; NORMAL_COLOR = "#12664F"

FIGSIZE = (12.3, 8.6); FIGURE_DPI = 320
OUTER_WIDTH_RATIOS = (0.29, 0.71); OUTER_HEIGHT_RATIOS = (0.20, 0.80)
OUTER_LEFT = 0.055; OUTER_RIGHT = 0.92; OUTER_BOTTOM = 0.065; OUTER_TOP = 0.95
OUTER_WSPACE = 0.18; OUTER_HSPACE = 0.20

LEFT_HEIGHT_RATIOS_FIRST = 0.72; LEFT_HSPACE = 0.18
PSD_WSPACE = 0.10
STFT_WSPACE = 0.08; STFT_HSPACE = 0.055
WAVEFORM_STFT_HEIGHT_RATIOS = (0.24, 0.76); WAVEFORM_STFT_HSPACE = 0.015

add_font = 2
FONT_SIZES = {
    "base": 9.2 + add_font,
    "axis_label": 9.8 + add_font,
    "axis_title": 10.2,
    "tick": 8.6,
    "legend": 8.5,
    "annotation": 9.0,
    "panel_label": 15.0 + add_font,
    "colorbar_label": 8.2,
    "colorbar_tick": 8.0,
}

PANEL_LABEL_POSITION = {"a": (-0.18, 1.18), "b": (-0.34, 1.18), "c": (-0.34, 1.85)}
STATE_LABEL_POSITION = (0.015, 0.78)

SEQUENCE_PCS_SIZE = 4.0; SEQUENCE_OTHER_SIZE = 2.1
SEQUENCE_PCS_ALPHA = 0.95; SEQUENCE_OTHER_ALPHA = 0.38
PCS_HORIZONTAL_LINEWIDTH = 0.8; PCS_HORIZONTAL_ALPHA = 0.45

CUMULATIVE_PCS_LINEWIDTH = 1.45; CUMULATIVE_OTHER_LINEWIDTH = 0.90
CUMULATIVE_PCS_ALPHA = 0.98; CUMULATIVE_OTHER_ALPHA = 0.60
OCCURRENCE_LINEWIDTH = 0.30; OCCURRENCE_PCS_ALPHA = 0.20; OCCURRENCE_OTHER_ALPHA = 0.10
CUMULATIVE_PCS_FILL_ALPHA = 0.075; CUMULATIVE_OTHER_FILL_ALPHA = 0.025
PCS_SPINE_LINEWIDTH = 1.05

GRID_COLOR = "#DDDDDD"
SEQUENCE_GRID_LINEWIDTH = 0.35; SEQUENCE_GRID_ALPHA = 0.65
CUMULATIVE_GRID_LINEWIDTH = 0.35; CUMULATIVE_GRID_ALPHA = 0.55

PSD_GRID_COLOR = "#EEEEEE"; PSD_GRID_LINEWIDTH = 0.32; PSD_GRID_ALPHA = 0.55
PSD_CMAP = "YlOrRd"; PSD_FREQUENCY_LIMIT = (0, 50)
PSD_MEDIAN_LINEWIDTH = 1.10; PSD_TITLE_PAD = 3; PSD_PCS_SPINE_LINEWIDTH = 1.0

WAVEFORM_COLOR = "#303030"; WAVEFORM_LINEWIDTH = 0.30
WAVEFORM_TIME_LIMIT = (0, 60); WAVEFORM_AMPLITUDE_LIMIT = (-1.1, 1.1)
WAVEFORM_NORMALIZATION_PERCENTILE = 99

STFT_FREQUENCY_LIMIT = (0, 50); STFT_NORMALIZATION_PERCENTILE = 99.5
STFT_NPERSEG = 512; STFT_NOVERLAP = 448
STFT_CMAP = "turbo"; STFT_VMIN = 0; STFT_VMAX = 1

NORMAL_SPINE_COLOR = "#777777"; NORMAL_SPINE_LINEWIDTH = 0.25
PCS_STFT_SPINE_LINEWIDTH = 0.55

CUMULATIVE_COUNT_LABEL_PAD = 2; CUMULATIVE_FRACTION_LABEL_PAD = 3
SECONDARY_AXIS_TICK_PAD = 1.5; SECONDARY_AXIS_TICK_LENGTH = 2.5
BOTTOM_TIME_LABEL_PAD = 1; LEFT_FREQUENCY_LABEL_PAD = 1

PSD_CB_XPAD = 0.008; PSD_CB_WIDTH = 0.007; PSD_CB_YSHRINK = 0.000
STFT_CB_XPAD = 0.008; STFT_CB_WIDTH = 0.007; STFT_CB_YSHRINK = 0.005

COLORBAR_LABEL_PAD = 3
COLORBAR_TICK_PAD = 1.5; COLORBAR_TICK_LENGTH = 2.5

SAVE_DPI = 320
PNG_NAME = f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.png"
SVG_NAME = f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.tiff"

cluster_colors = plt.get_cmap(CLUSTER_CMAP)(
    np.linspace(0, 1, max(n_clusters, 3))
)
pcs_color = PCS_COLOR

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": FONT_SIZES["base"],
    "axes.linewidth": 0.75,
    "axes.labelsize": FONT_SIZES["axis_label"],
    "axes.titlesize": FONT_SIZES["axis_title"],
    "xtick.labelsize": FONT_SIZES["tick"],
    "ytick.labelsize": FONT_SIZES["tick"],
    "legend.fontsize": FONT_SIZES["legend"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


fig = plt.figure(figsize=FIGSIZE, dpi=FIGURE_DPI)

outer = GridSpec(
    2,
    2,
    figure=fig,
    width_ratios=OUTER_WIDTH_RATIOS,
    height_ratios=OUTER_HEIGHT_RATIOS,
    left=OUTER_LEFT,
    right=OUTER_RIGHT,
    bottom=OUTER_BOTTOM,
    top=OUTER_TOP,
    wspace=OUTER_WSPACE,
    hspace=OUTER_HSPACE,
)

left = GridSpecFromSubplotSpec(
    n_clusters + 1,
    1,
    subplot_spec=outer[:, 0],
    height_ratios=[LEFT_HEIGHT_RATIOS_FIRST] + [1] * n_clusters,
    hspace=LEFT_HSPACE,
)

ax_seq = fig.add_subplot(left[0, 0])

for cid in clusters:
    idx = labels == cid
    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]

    ax_seq.scatter(
        time_h[idx],
        np.full(idx.sum(), cid),
        s=(
            SEQUENCE_PCS_SIZE
            if cid == pcs_cluster
            else SEQUENCE_OTHER_SIZE
        ),
        color=color,
        alpha=(
            SEQUENCE_PCS_ALPHA
            if cid == pcs_cluster
            else SEQUENCE_OTHER_ALPHA
        ),
        linewidths=0,
    )

if pcs_cluster is not None:
    ax_seq.axhline(
        pcs_cluster,
        color=pcs_color,
        lw=PCS_HORIZONTAL_LINEWIDTH,
        alpha=PCS_HORIZONTAL_ALPHA,
    )

ax_seq.set_xlim(-WINDOW_HOURS, 0)
ax_seq.set_yticks(clusters)
ax_seq.set_ylabel("DEC state")
ax_seq.tick_params(labelbottom=False)

ax_seq.grid(
    True,
    color=GRID_COLOR,
    lw=SEQUENCE_GRID_LINEWIDTH,
    alpha=SEQUENCE_GRID_ALPHA,
)

ax_seq.text(
    *PANEL_LABEL_POSITION["a"],
    "a",
    transform=ax_seq.transAxes,
    fontsize=FONT_SIZES["panel_label"],
    fontweight="bold",
    va="top",
)


for i, cid in enumerate(clusters):
    ax = fig.add_subplot(left[i + 1, 0], sharex=ax_seq)

    hit = (labels == cid).astype(int)
    cum = np.cumsum(hit)

    n_state = int(hit.sum())
    n_scale = max(n_state, 1)

    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]

    line_alpha = (
        CUMULATIVE_PCS_ALPHA
        if cid == pcs_cluster
        else CUMULATIVE_OTHER_ALPHA
    )

    line_width = (
        CUMULATIVE_PCS_LINEWIDTH
        if cid == pcs_cluster
        else CUMULATIVE_OTHER_LINEWIDTH
    )

    for x in time_h[hit == 1]:
        ax.axvline(
            x,
            color=color,
            lw=OCCURRENCE_LINEWIDTH,
            alpha=(
                OCCURRENCE_PCS_ALPHA
                if cid == pcs_cluster
                else OCCURRENCE_OTHER_ALPHA
            ),
        )

    ax.plot(
        time_h,
        cum,
        color=color,
        lw=line_width,
        alpha=line_alpha,
    )

    ax.fill_between(
        time_h,
        0,
        cum,
        color=color,
        alpha=(
            CUMULATIVE_PCS_FILL_ALPHA
            if cid == pcs_cluster
            else CUMULATIVE_OTHER_FILL_ALPHA
        ),
    )

    ax.set_ylim(0, n_scale)
    ax.set_ylabel(
        "Cum. count",
        labelpad=CUMULATIVE_COUNT_LABEL_PAD,
    )

    ax.grid(
        True,
        color=GRID_COLOR,
        lw=CUMULATIVE_GRID_LINEWIDTH,
        alpha=CUMULATIVE_GRID_ALPHA,
    )

    ax_prob = ax.secondary_yaxis(
        "right",
        functions=(
            lambda count, n=n_scale: count / n,
            lambda fraction, n=n_scale: fraction * n,
        ),
    )

    ax_prob.set_yticks([0.0, 0.5, 1.0])
    ax_prob.set_yticklabels(["0", "0.5", "1"])

    ax_prob.tick_params(
        axis="y",
        labelsize=FONT_SIZES["tick"],
        pad=SECONDARY_AXIS_TICK_PAD,
        length=SECONDARY_AXIS_TICK_LENGTH,
    )

    if i == n_clusters // 2:
        ax_prob.set_ylabel(
            "Probability",
            fontsize=FONT_SIZES["axis_label"],
            labelpad=CUMULATIVE_FRACTION_LABEL_PAD,
        )

    ax.text(
        *STATE_LABEL_POSITION,
        f"C{cid}" + (" | PCS" if cid == pcs_cluster else ""),
        transform=ax.transAxes,
        color=pcs_color if cid == pcs_cluster else "#555555",
        fontsize=FONT_SIZES["annotation"],
        fontweight="bold" if cid == pcs_cluster else "normal",
    )

    if i < n_clusters - 1:
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel("Time before debris-flow initiation (h)")

    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color)
            spine.set_linewidth(PCS_SPINE_LINEWIDTH)


# ===========================================================================
# b. PSD probability-density distributions
# ===========================================================================

psd_grid = GridSpecFromSubplotSpec(
    1,
    n_clusters,
    subplot_spec=outer[0, 1],
    wspace=PSD_WSPACE,
)

psd_axes = []
pdf_mappable = None

for j, cid in enumerate(clusters):
    ax = fig.add_subplot(psd_grid[0, j])
    psd_axes.append(ax)

    den = density_by_cluster[cid]

    pdf_mappable = ax.pcolormesh(
        freq_bins,
        psd_bins,
        den.T,
        cmap=PSD_CMAP,
        shading="auto",
        vmin=0,
        vmax=vmax_pdf,
        rasterized=True,
    )

    median_psd = np.nanmedian(
        psd_by_cluster[cid],
        axis=0,
    )

    ax.plot(
        freq_ref,
        median_psd,
        color=pcs_color if cid == pcs_cluster else NORMAL_COLOR,
        lw=PSD_MEDIAN_LINEWIDTH,
    )

    ax.set_xlim(*PSD_FREQUENCY_LIMIT)
    ax.set_ylim(psd_min, psd_max)

    ax.set_title(
        f"C{cid}"
        + (" | PCS" if cid == pcs_cluster else "")
        + f"\nn={(labels == cid).sum()}",
        color=pcs_color if cid == pcs_cluster else "black",
        fontweight="bold" if cid == pcs_cluster else "normal",
        pad=PSD_TITLE_PAD,
    )

    ax.set_xlabel("Frequency (Hz)")

    ax.grid(
        True,
        color=PSD_GRID_COLOR,
        lw=PSD_GRID_LINEWIDTH,
        alpha=PSD_GRID_ALPHA,
    )

    if j == 0:
        ax.set_ylabel("PSD (dB)")

        ax.text(
            *PANEL_LABEL_POSITION["b"],
            "b",
            transform=ax.transAxes,
            fontsize=FONT_SIZES["panel_label"],
            fontweight="bold",
            va="top",
        )
    else:
        ax.tick_params(labelleft=False)

    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color)
            spine.set_linewidth(PSD_PCS_SPINE_LINEWIDTH)


# ===========================================================================
# c. First seven representative waveform and STFT samples
# ===========================================================================

stft_grid = GridSpecFromSubplotSpec(
    N_REPRESENTATIVES,
    n_clusters,
    subplot_spec=outer[1, 1],
    wspace=STFT_WSPACE,
    hspace=STFT_HSPACE,
)

stft_axes = []
last_im = None

for j, cid in enumerate(clusters):
    reps = rep_by_cluster[cid][:N_REPRESENTATIVES]

    for r in range(N_REPRESENTATIVES):
        cell = GridSpecFromSubplotSpec(
            2,
            1,
            subplot_spec=stft_grid[r, j],
            height_ratios=WAVEFORM_STFT_HEIGHT_RATIOS,
            hspace=WAVEFORM_STFT_HSPACE,
        )

        ax_w = fig.add_subplot(cell[0, 0])
        ax_s = fig.add_subplot(cell[1, 0])

        stft_axes.append(ax_s)

        if r < len(reps):
            x = waveform_for_index(reps[r])
            t = np.arange(len(x)) / fs

            scale = np.nanpercentile(
                np.abs(x),
                WAVEFORM_NORMALIZATION_PERCENTILE,
            )

            x_norm = x / scale if scale > 0 else x

            ax_w.plot(
                t,
                x_norm,
                color=WAVEFORM_COLOR,
                lw=WAVEFORM_LINEWIDTH,
            )

            nperseg_use = min(STFT_NPERSEG, len(x))
            noverlap_use = min(
                STFT_NOVERLAP,
                max(0, nperseg_use - 1),
            )

            f, tt, zz = stft(
                x,
                fs=fs,
                nperseg=nperseg_use,
                noverlap=noverlap_use,
            )

            keep = f <= STFT_FREQUENCY_LIMIT[1]
            amp = np.abs(zz[keep])

            amp_scale = (
                np.nanpercentile(
                    amp,
                    STFT_NORMALIZATION_PERCENTILE,
                )
                + 1e-12
            )

            amp = np.clip(
                amp / amp_scale,
                STFT_VMIN,
                STFT_VMAX,
            )

            last_im = ax_s.pcolormesh(
                tt,
                f[keep],
                amp,
                cmap=STFT_CMAP,
                shading="auto",
                vmin=STFT_VMIN,
                vmax=STFT_VMAX,
                rasterized=True,
            )

        ax_w.set_xlim(*WAVEFORM_TIME_LIMIT)
        ax_w.set_ylim(*WAVEFORM_AMPLITUDE_LIMIT)

        ax_s.set_xlim(*WAVEFORM_TIME_LIMIT)
        ax_s.set_ylim(*STFT_FREQUENCY_LIMIT)

        for ax in (ax_w, ax_s):
            ax.set_xticks([])
            ax.set_yticks([])

            for spine in ax.spines.values():
                spine.set_linewidth(NORMAL_SPINE_LINEWIDTH)
                spine.set_color(NORMAL_SPINE_COLOR)

        if cid == pcs_cluster:
            for ax in (ax_w, ax_s):
                for spine in ax.spines.values():
                    spine.set_color(pcs_color)
                    spine.set_linewidth(
                        PCS_STFT_SPINE_LINEWIDTH
                    )

        if j == 0 and r == 0:
            ax_w.text(
                *PANEL_LABEL_POSITION["c"],
                "c",
                transform=ax_w.transAxes,
                fontsize=FONT_SIZES["panel_label"],
                fontweight="bold",
                va="top",
            )

        if r == N_REPRESENTATIVES - 1:
            ax_s.set_xticks([0, 30, 60])
            ax_s.set_xlabel(
                "Time (s)",
                labelpad=BOTTOM_TIME_LABEL_PAD,
            )

        # 第1列和第4列显示频率刻度及y轴标题
        if j in (0, 3):
            ax_s.set_yticks([0, 25, 50])
            ax_s.set_ylabel(
                "Frequency (Hz)",
                labelpad=LEFT_FREQUENCY_LABEL_PAD,
            )


# ===========================================================================
# Independent colorbars
# ===========================================================================

fig.canvas.draw()

bbox_psd = Bbox.union([
    ax.get_position()
    for ax in psd_axes
])

cax_pdf = fig.add_axes([
    bbox_psd.x1 + PSD_CB_XPAD,
    bbox_psd.y0 + PSD_CB_YSHRINK,
    PSD_CB_WIDTH,
    bbox_psd.height - 2 * PSD_CB_YSHRINK,
])

cb = fig.colorbar(
    pdf_mappable,
    cax=cax_pdf,
    orientation="vertical",
)

cb.set_label(
    "Probability density (%)",
    fontsize=FONT_SIZES["colorbar_label"],
    labelpad=COLORBAR_LABEL_PAD,
)

cb.ax.tick_params(
    labelsize=FONT_SIZES["colorbar_tick"],
    pad=COLORBAR_TICK_PAD,
    length=COLORBAR_TICK_LENGTH,
)


bbox_stft = Bbox.union([
    ax.get_position()
    for ax in stft_axes
])

cax_stft = fig.add_axes([
    bbox_stft.x1 + STFT_CB_XPAD,
    bbox_stft.y0 + STFT_CB_YSHRINK,
    STFT_CB_WIDTH,
    bbox_stft.height - 2 * STFT_CB_YSHRINK,
])

cb2 = fig.colorbar(
    last_im,
    cax=cax_stft,
    orientation="vertical",
)

cb2.set_label(
    "Normalized amplitude",
    fontsize=FONT_SIZES["colorbar_label"],
    labelpad=COLORBAR_LABEL_PAD,
)

cb2.ax.tick_params(
    labelsize=FONT_SIZES["colorbar_tick"],
    pad=COLORBAR_TICK_PAD,
    length=COLORBAR_TICK_LENGTH,
)


# ===========================================================================
# Save
# ===========================================================================

out_png = OUT_DIR / PNG_NAME
out_svg = OUT_DIR / SVG_NAME

fig.savefig(
    out_png,
    dpi=SAVE_DPI,
    bbox_inches="tight",
    facecolor="white",
)

fig.savefig(
    out_svg,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

print(f"PNG saved to: {out_png}")

In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.transforms import Bbox
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import stft


# ========================== EDIT FIGURE STYLE HERE ==========================
N_REPRESENTATIVES = 7; CLUSTER_CMAP = "tab10"; PCS_COLOR = "#D95F02"; NORMAL_COLOR = "#12664F"

FIGSIZE = (12.3, 8.6); FIGURE_DPI = 320
OUTER_WIDTH_RATIOS = (0.29, 0.71); OUTER_HEIGHT_RATIOS = (0.20, 0.80)
OUTER_LEFT = 0.055; OUTER_RIGHT = 0.92; OUTER_BOTTOM = 0.065; OUTER_TOP = 0.95
OUTER_WSPACE = 0.20; OUTER_HSPACE = 0.20

LEFT_HEIGHT_RATIOS_FIRST = 0.72; LEFT_HSPACE = 0.18; PSD_WSPACE = 0.10
STFT_WSPACE = 0.08; STFT_HSPACE = 0.055
WAVEFORM_STFT_HEIGHT_RATIOS = (0.24, 0.76); WAVEFORM_STFT_HSPACE = 0.015

add_font = 2; XYTICK_SIZE = 10
FONT_SIZES = {
    "base": 9.2 + add_font, "axis_label": 9.8 + add_font, "axis_title": 10.2,
    "tick": XYTICK_SIZE, "legend": 9.5, "annotation": 9.0,
    "panel_label": 15.0 + add_font, "colorbar_label": 10.9, "colorbar_tick": 9.5
}

PANEL_LABEL_POSITION = {"a": (-0.18, 1.31), "b": (-0.34, 1.18), "c": (-0.34, 1.85)}
STATE_LABEL_POSITION = (0.015, 0.78)

SEQUENCE_PCS_SIZE = 4.0; SEQUENCE_OTHER_SIZE = 2.1
SEQUENCE_PCS_ALPHA = 0.95; SEQUENCE_OTHER_ALPHA = 0.38
PCS_HORIZONTAL_LINEWIDTH = 0.8; PCS_HORIZONTAL_ALPHA = 0.45

CUMULATIVE_PCS_LINEWIDTH = 1.45; CUMULATIVE_OTHER_LINEWIDTH = 0.90
CUMULATIVE_PCS_ALPHA = 0.98; CUMULATIVE_OTHER_ALPHA = 0.60
OCCURRENCE_LINEWIDTH = 0.30; OCCURRENCE_PCS_ALPHA = 0.20; OCCURRENCE_OTHER_ALPHA = 0.10
CUMULATIVE_PCS_FILL_ALPHA = 0.075; CUMULATIVE_OTHER_FILL_ALPHA = 0.025
PCS_SPINE_LINEWIDTH = 1.05

GRID_COLOR = "#DDDDDD"
SEQUENCE_GRID_LINEWIDTH = 0.35; SEQUENCE_GRID_ALPHA = 0.65
CUMULATIVE_GRID_LINEWIDTH = 0.35; CUMULATIVE_GRID_ALPHA = 0.55

PSD_GRID_COLOR = "#EEEEEE"; PSD_GRID_LINEWIDTH = 0.32; PSD_GRID_ALPHA = 0.55
PSD_CMAP = "YlOrRd"; PSD_FREQUENCY_LIMIT = (0, 50)
PSD_MEDIAN_LINEWIDTH = 1.10; PSD_TITLE_PAD = 3; PSD_PCS_SPINE_LINEWIDTH = 1.0

WAVEFORM_COLOR = "#303030"; WAVEFORM_LINEWIDTH = 0.30
WAVEFORM_TIME_LIMIT = (0, 60); WAVEFORM_AMPLITUDE_LIMIT = (-1.1, 1.1)
WAVEFORM_NORMALIZATION_PERCENTILE = 99

STFT_FREQUENCY_LIMIT = (0, 50); STFT_NORMALIZATION_PERCENTILE = 99.5
STFT_NPERSEG = 512; STFT_NOVERLAP = 448
STFT_CMAP = "turbo"; STFT_VMIN = 0; STFT_VMAX = 1

NORMAL_SPINE_COLOR = "#777777"; NORMAL_SPINE_LINEWIDTH = 0.25
PCS_STFT_SPINE_LINEWIDTH = 0.55

CUMULATIVE_COUNT_LABEL_PAD = 2; CUMULATIVE_FRACTION_LABEL_PAD = 3
SECONDARY_AXIS_TICK_PAD = 1.5; SECONDARY_AXIS_TICK_LENGTH = 2.5
BOTTOM_TIME_LABEL_PAD = 1; LEFT_FREQUENCY_LABEL_PAD = 1

PSD_CB_XPAD = 0.008; PSD_CB_WIDTH = 0.007; PSD_CB_YSHRINK = 0.000
STFT_CB_XPAD = 0.008; STFT_CB_WIDTH = 0.007; STFT_CB_YSHRINK = 0.005
COLORBAR_LABEL_PAD = 3; COLORBAR_TICK_PAD = 1.5; COLORBAR_TICK_LENGTH = 2.5

SAVE_DPI = 320
PNG_NAME = f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.png"
SVG_NAME = f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.tiff"
# ===========================================================================


cluster_colors = plt.get_cmap(CLUSTER_CMAP)(np.linspace(0, 1, max(n_clusters, 3)))
pcs_color = PCS_COLOR

mpl.rcParams.update({
    "font.family": "Arial", "font.size": FONT_SIZES["base"], "axes.linewidth": 0.75,
    "axes.labelsize": FONT_SIZES["axis_label"], "axes.titlesize": FONT_SIZES["axis_title"],
    "xtick.labelsize": FONT_SIZES["tick"], "ytick.labelsize": FONT_SIZES["tick"],
    "legend.fontsize": FONT_SIZES["legend"], "pdf.fonttype": 42,
    "ps.fonttype": 42, "svg.fonttype": "none"
})

fig = plt.figure(figsize=FIGSIZE, dpi=FIGURE_DPI)
outer = GridSpec(
    2, 2, figure=fig, width_ratios=OUTER_WIDTH_RATIOS, height_ratios=OUTER_HEIGHT_RATIOS,
    left=OUTER_LEFT, right=OUTER_RIGHT, bottom=OUTER_BOTTOM, top=OUTER_TOP,
    wspace=OUTER_WSPACE, hspace=OUTER_HSPACE
)


# a. DEC sequence and cumulative occurrence curves
left = GridSpecFromSubplotSpec(
    n_clusters + 1, 1, subplot_spec=outer[:, 0],
    height_ratios=[LEFT_HEIGHT_RATIOS_FIRST] + [1] * n_clusters,
    hspace=LEFT_HSPACE
)
ax_seq = fig.add_subplot(left[0, 0])

for cid in clusters:
    idx = labels == cid
    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]
    ax_seq.scatter(
        time_h[idx], np.full(idx.sum(), cid),
        s=SEQUENCE_PCS_SIZE if cid == pcs_cluster else SEQUENCE_OTHER_SIZE,
        color=color,
        alpha=SEQUENCE_PCS_ALPHA if cid == pcs_cluster else SEQUENCE_OTHER_ALPHA,
        linewidths=0
    )

if pcs_cluster is not None:
    ax_seq.axhline(pcs_cluster, color=pcs_color, lw=PCS_HORIZONTAL_LINEWIDTH, alpha=PCS_HORIZONTAL_ALPHA)

ax_seq.set_xlim(-WINDOW_HOURS, 0); ax_seq.set_yticks(clusters); ax_seq.set_ylabel("DEC state")
ax_seq.tick_params(labelbottom=False)
ax_seq.grid(True, color=GRID_COLOR, lw=SEQUENCE_GRID_LINEWIDTH, alpha=SEQUENCE_GRID_ALPHA)
ax_seq.text(
    *PANEL_LABEL_POSITION["a"], "a", transform=ax_seq.transAxes,
    fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top"
)

for i, cid in enumerate(clusters):
    ax = fig.add_subplot(left[i + 1, 0], sharex=ax_seq)
    hit = (labels == cid).astype(int); cum = np.cumsum(hit)
    n_state = int(hit.sum()); n_scale = max(n_state, 1)
    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]
    line_alpha = CUMULATIVE_PCS_ALPHA if cid == pcs_cluster else CUMULATIVE_OTHER_ALPHA
    line_width = CUMULATIVE_PCS_LINEWIDTH if cid == pcs_cluster else CUMULATIVE_OTHER_LINEWIDTH

    for x in time_h[hit == 1]:
        ax.axvline(
            x, color=color, lw=OCCURRENCE_LINEWIDTH,
            alpha=OCCURRENCE_PCS_ALPHA if cid == pcs_cluster else OCCURRENCE_OTHER_ALPHA
        )

    ax.plot(time_h, cum, color=color, lw=line_width, alpha=line_alpha)
    ax.fill_between(
        time_h, 0, cum, color=color,
        alpha=CUMULATIVE_PCS_FILL_ALPHA if cid == pcs_cluster else CUMULATIVE_OTHER_FILL_ALPHA
    )
    ax.set_ylim(0, n_scale); ax.set_ylabel("Cum. count", labelpad=CUMULATIVE_COUNT_LABEL_PAD)
    ax.grid(True, color=GRID_COLOR, lw=CUMULATIVE_GRID_LINEWIDTH, alpha=CUMULATIVE_GRID_ALPHA)

    ax_prob = ax.secondary_yaxis(
        "right",
        functions=(
            lambda count, n=n_scale: count / n,
            lambda fraction, n=n_scale: fraction * n
        )
    )
    ax_prob.set_yticks([0.0, 0.5, 1.0]); ax_prob.set_yticklabels(["0", "0.5", "1"])
    ax_prob.tick_params(
        axis="y", labelsize=FONT_SIZES["tick"],
        pad=SECONDARY_AXIS_TICK_PAD, length=SECONDARY_AXIS_TICK_LENGTH
    )

    if i == n_clusters // 2:
        ax_prob.set_ylabel(
            "Probability", fontsize=FONT_SIZES["axis_label"],
            labelpad=CUMULATIVE_FRACTION_LABEL_PAD
        )

    ax.text(
        *STATE_LABEL_POSITION, f"C{cid}" + (" | PCS" if cid == pcs_cluster else ""),
        transform=ax.transAxes, color=pcs_color if cid == pcs_cluster else "#555555",
        fontsize=FONT_SIZES["annotation"],
        fontweight="bold" if cid == pcs_cluster else "normal"
    )

    if i < n_clusters - 1:
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel("Time before debris-flow initiation (h)")

    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color); spine.set_linewidth(PCS_SPINE_LINEWIDTH)


# b. PSD probability-density distributions
psd_grid = GridSpecFromSubplotSpec(1, n_clusters, subplot_spec=outer[0, 1], wspace=PSD_WSPACE)
psd_axes = []; pdf_mappable = None

for j, cid in enumerate(clusters):
    ax = fig.add_subplot(psd_grid[0, j]); psd_axes.append(ax)
    den = density_by_cluster[cid]

    pdf_mappable = ax.pcolormesh(
        freq_bins, psd_bins, den.T, cmap=PSD_CMAP, shading="auto",
        vmin=0, vmax=vmax_pdf, rasterized=True
    )

    median_psd = np.nanmedian(psd_by_cluster[cid], axis=0)
    ax.plot(
        freq_ref, median_psd,
        color=pcs_color if cid == pcs_cluster else NORMAL_COLOR,
        lw=PSD_MEDIAN_LINEWIDTH
    )

    ax.set_xlim(*PSD_FREQUENCY_LIMIT); ax.set_ylim(psd_min, psd_max)
    ax.set_title(
        f"C{cid}" + (" | PCS" if cid == pcs_cluster else "") + f"\nn={(labels == cid).sum()}",
        color=pcs_color if cid == pcs_cluster else "black",
        fontweight="bold" if cid == pcs_cluster else "normal",
        pad=PSD_TITLE_PAD
    )
    ax.set_xlabel("Frequency (Hz)")
    ax.grid(True, color=PSD_GRID_COLOR, lw=PSD_GRID_LINEWIDTH, alpha=PSD_GRID_ALPHA)

    if j == 0:
        ax.set_ylabel("PSD (dB)")
        ax.text(
            *PANEL_LABEL_POSITION["b"], "b", transform=ax.transAxes,
            fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top"
        )
    else:
        ax.tick_params(labelleft=False)

    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color); spine.set_linewidth(PSD_PCS_SPINE_LINEWIDTH)


# c. Representative waveform and STFT samples
stft_grid = GridSpecFromSubplotSpec(
    N_REPRESENTATIVES, n_clusters, subplot_spec=outer[1, 1],
    wspace=STFT_WSPACE, hspace=STFT_HSPACE
)
stft_axes = []; last_im = None

for j, cid in enumerate(clusters):
    reps = rep_by_cluster[cid][:N_REPRESENTATIVES]

    for r in range(N_REPRESENTATIVES):
        cell = GridSpecFromSubplotSpec(
            2, 1, subplot_spec=stft_grid[r, j],
            height_ratios=WAVEFORM_STFT_HEIGHT_RATIOS,
            hspace=WAVEFORM_STFT_HSPACE
        )
        ax_w = fig.add_subplot(cell[0, 0]); ax_s = fig.add_subplot(cell[1, 0])
        stft_axes.append(ax_s)

        if r < len(reps):
            x = waveform_for_index(reps[r]); t = np.arange(len(x)) / fs
            scale = np.nanpercentile(np.abs(x), WAVEFORM_NORMALIZATION_PERCENTILE)
            x_norm = x / scale if scale > 0 else x
            ax_w.plot(t, x_norm, color=WAVEFORM_COLOR, lw=WAVEFORM_LINEWIDTH)

            nperseg_use = min(STFT_NPERSEG, len(x))
            noverlap_use = min(STFT_NOVERLAP, max(0, nperseg_use - 1))
            f, tt, zz = stft(x, fs=fs, nperseg=nperseg_use, noverlap=noverlap_use)

            keep = f <= STFT_FREQUENCY_LIMIT[1]
            amp = np.abs(zz[keep])
            amp_scale = np.nanpercentile(amp, STFT_NORMALIZATION_PERCENTILE) + 1e-12
            amp = np.clip(amp / amp_scale, STFT_VMIN, STFT_VMAX)

            last_im = ax_s.pcolormesh(
                tt, f[keep], amp, cmap=STFT_CMAP, shading="auto",
                vmin=STFT_VMIN, vmax=STFT_VMAX, rasterized=True
            )

        ax_w.set_xlim(*WAVEFORM_TIME_LIMIT); ax_w.set_ylim(*WAVEFORM_AMPLITUDE_LIMIT)
        ax_s.set_xlim(*WAVEFORM_TIME_LIMIT); ax_s.set_ylim(*STFT_FREQUENCY_LIMIT)

        for ax in (ax_w, ax_s):
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(NORMAL_SPINE_LINEWIDTH); spine.set_color(NORMAL_SPINE_COLOR)

        if cid == pcs_cluster:
            for ax in (ax_w, ax_s):
                for spine in ax.spines.values():
                    spine.set_color(pcs_color); spine.set_linewidth(PCS_STFT_SPINE_LINEWIDTH)

        if j == 0 and r == 0:
            ax_w.text(
                *PANEL_LABEL_POSITION["c"], "c", transform=ax_w.transAxes,
                fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top"
            )

        if r == N_REPRESENTATIVES - 1:
            ax_s.set_xticks([0, 30, 60])
            ax_s.set_xlabel("Time (s)", labelpad=BOTTOM_TIME_LABEL_PAD)

        if j == 0:
            ax_s.set_yticks([0, 25, 50])
            if r == N_REPRESENTATIVES // 2:
                ax_s.set_ylabel("Frequency (Hz)", labelpad=LEFT_FREQUENCY_LABEL_PAD)


# Independent colorbars
fig.canvas.draw()

bbox_psd = Bbox.union([ax.get_position() for ax in psd_axes])
cax_pdf = fig.add_axes([
    bbox_psd.x1 + PSD_CB_XPAD, bbox_psd.y0 + PSD_CB_YSHRINK,
    PSD_CB_WIDTH, bbox_psd.height - 2 * PSD_CB_YSHRINK
])
cb = fig.colorbar(pdf_mappable, cax=cax_pdf, orientation="vertical")
cb.set_label(
    "Probability density (%)",
    fontsize=FONT_SIZES["colorbar_label"],
    labelpad=COLORBAR_LABEL_PAD
)
cb.ax.tick_params(
    labelsize=FONT_SIZES["colorbar_tick"],
    pad=COLORBAR_TICK_PAD,
    length=COLORBAR_TICK_LENGTH
)

bbox_stft = Bbox.union([ax.get_position() for ax in stft_axes])
cax_stft = fig.add_axes([
    bbox_stft.x1 + STFT_CB_XPAD, bbox_stft.y0 + STFT_CB_YSHRINK,
    STFT_CB_WIDTH, bbox_stft.height - 2 * STFT_CB_YSHRINK
])
cb2 = fig.colorbar(last_im, cax=cax_stft, orientation="vertical")
cb2.set_label(
    "Normalized amplitude",
    fontsize=FONT_SIZES["colorbar_label"],
    labelpad=COLORBAR_LABEL_PAD
)
cb2.ax.tick_params(
    labelsize=FONT_SIZES["colorbar_tick"],
    pad=COLORBAR_TICK_PAD,
    length=COLORBAR_TICK_LENGTH
)


# Save
out_png = OUT_DIR / PNG_NAME; out_svg = OUT_DIR / SVG_NAME
fig.savefig(out_png, dpi=SAVE_DPI, bbox_inches="tight", facecolor="white")
fig.savefig(out_svg, bbox_inches="tight", facecolor="white")
plt.show()
print(f"PNG saved to: {out_png}")

In [ ]:
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.transforms import Bbox
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import stft


# ========================== EDIT FIGURE STYLE HERE ==========================
N_REPRESENTATIVES = 7; CLUSTER_CMAP = "tab10"; PCS_COLOR = "#D95F02"; NORMAL_COLOR = "#12664F"

FIGSIZE = (12.3, 8.6); FIGURE_DPI = 320
OUTER_WIDTH_RATIOS = (0.29, 0.71); OUTER_HEIGHT_RATIOS = (0.20, 0.80)
OUTER_LEFT = 0.055; OUTER_RIGHT = 0.92; OUTER_BOTTOM = 0.065; OUTER_TOP = 0.95
OUTER_WSPACE = 0.20; OUTER_HSPACE = 0.20

LEFT_HEIGHT_RATIOS_FIRST = 0.72; LEFT_HSPACE = 0.18; PSD_WSPACE = 0.10
STFT_WSPACE = 0.08; STFT_HSPACE = 0.055
WAVEFORM_STFT_HEIGHT_RATIOS = (0.24, 0.76); WAVEFORM_STFT_HSPACE = 0.015

add_font = 2; XYTICK_SIZE = 10
FONT_SIZES = {
    "base": 9.2 + add_font, "axis_label": 9.8 + add_font, "axis_title": 10.2,
    "tick": XYTICK_SIZE, "legend": 9.5, "annotation": 9.0,
    "panel_label": 15.0 + add_font, "colorbar_label": 10.9, "colorbar_tick": 9.5
}

PANEL_LABEL_POSITION = {"a": (-0.18, 1.31), "b": (-0.34, 1.18), "c": (-0.34, 1.85)}
STATE_LABEL_POSITION = (0.015, 0.78)

SEQUENCE_PCS_SIZE = 4.0; SEQUENCE_OTHER_SIZE = 2.1
SEQUENCE_PCS_ALPHA = 0.95; SEQUENCE_OTHER_ALPHA = 0.38
PCS_HORIZONTAL_LINEWIDTH = 0.8; PCS_HORIZONTAL_ALPHA = 0.45

CUMULATIVE_PCS_LINEWIDTH = 1.45; CUMULATIVE_OTHER_LINEWIDTH = 0.90
CUMULATIVE_PCS_ALPHA = 0.98; CUMULATIVE_OTHER_ALPHA = 0.60
OCCURRENCE_LINEWIDTH = 0.30; OCCURRENCE_PCS_ALPHA = 0.20; OCCURRENCE_OTHER_ALPHA = 0.10
CUMULATIVE_PCS_FILL_ALPHA = 0.075; CUMULATIVE_OTHER_FILL_ALPHA = 0.025
PCS_SPINE_LINEWIDTH = 1.05

GRID_COLOR = "#DDDDDD"
SEQUENCE_GRID_LINEWIDTH = 0.35; SEQUENCE_GRID_ALPHA = 0.65
CUMULATIVE_GRID_LINEWIDTH = 0.35; CUMULATIVE_GRID_ALPHA = 0.55

PSD_GRID_COLOR = "#EEEEEE"; PSD_GRID_LINEWIDTH = 0.32; PSD_GRID_ALPHA = 0.55
PSD_CMAP = "YlOrRd"; PSD_FREQUENCY_LIMIT = (0, 50)
PSD_MEDIAN_LINEWIDTH = 1.10; PSD_TITLE_PAD = 3; PSD_PCS_SPINE_LINEWIDTH = 1.0

WAVEFORM_COLOR = "#303030"; WAVEFORM_LINEWIDTH = 0.30
WAVEFORM_TIME_LIMIT = (0, 60); WAVEFORM_AMPLITUDE_LIMIT = (-1.1, 1.1)
WAVEFORM_NORMALIZATION_PERCENTILE = 99

STFT_FREQUENCY_LIMIT = (0, 50); STFT_NORMALIZATION_PERCENTILE = 99.5
STFT_NPERSEG = 512; STFT_NOVERLAP = 448
STFT_CMAP = "turbo"; STFT_VMIN = 0; STFT_VMAX = 1

NORMAL_SPINE_COLOR = "#777777"; NORMAL_SPINE_LINEWIDTH = 0.25
PCS_STFT_SPINE_LINEWIDTH = 0.55

CUMULATIVE_COUNT_LABEL_PAD = 2; CUMULATIVE_FRACTION_LABEL_PAD = 3
SECONDARY_AXIS_TICK_PAD = 1.5; SECONDARY_AXIS_TICK_LENGTH = 2.5
BOTTOM_TIME_LABEL_PAD = 1; LEFT_FREQUENCY_LABEL_PAD = 1

PSD_CB_XPAD = 0.008; PSD_CB_WIDTH = 0.007; PSD_CB_YSHRINK = 0.000
STFT_CB_XPAD = 0.008; STFT_CB_WIDTH = 0.007; STFT_CB_YSHRINK = 0.005
COLORBAR_LABEL_PAD = 3; COLORBAR_TICK_PAD = 1.5; COLORBAR_TICK_LENGTH = 2.5

SAVE_DPI = 320
PNG_NAME = f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.png"
SVG_NAME = f"figure4_cluster_diagnostics_{EVENT}_pcsC{pcs_cluster}_top8.tiff"
# ===========================================================================


cluster_colors = plt.get_cmap(CLUSTER_CMAP)(np.linspace(0, 1, max(n_clusters, 3)))
pcs_color = PCS_COLOR

mpl.rcParams.update({
    "font.family": "Arial", "font.size": FONT_SIZES["base"], "axes.linewidth": 0.75,
    "axes.labelsize": FONT_SIZES["axis_label"], "axes.titlesize": FONT_SIZES["axis_title"],
    "xtick.labelsize": FONT_SIZES["tick"], "ytick.labelsize": FONT_SIZES["tick"],
    "legend.fontsize": FONT_SIZES["legend"], "pdf.fonttype": 42,
    "ps.fonttype": 42, "svg.fonttype": "none"
})

fig = plt.figure(figsize=FIGSIZE, dpi=FIGURE_DPI)
outer = GridSpec(
    2, 2, figure=fig, width_ratios=OUTER_WIDTH_RATIOS, height_ratios=OUTER_HEIGHT_RATIOS,
    left=OUTER_LEFT, right=OUTER_RIGHT, bottom=OUTER_BOTTOM, top=OUTER_TOP,
    wspace=OUTER_WSPACE, hspace=OUTER_HSPACE
)


# a. DEC sequence and cumulative occurrence curves
left = GridSpecFromSubplotSpec(
    n_clusters + 1, 1, subplot_spec=outer[:, 0],
    height_ratios=[LEFT_HEIGHT_RATIOS_FIRST] + [1] * n_clusters,
    hspace=LEFT_HSPACE
)
ax_seq = fig.add_subplot(left[0, 0])

for cid in clusters:
    idx = labels == cid
    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]
    ax_seq.scatter(
        time_h[idx], np.full(idx.sum(), cid),
        s=SEQUENCE_PCS_SIZE if cid == pcs_cluster else SEQUENCE_OTHER_SIZE,
        color=color,
        alpha=SEQUENCE_PCS_ALPHA if cid == pcs_cluster else SEQUENCE_OTHER_ALPHA,
        linewidths=0
    )

if pcs_cluster is not None:
    ax_seq.axhline(pcs_cluster, color=pcs_color, lw=PCS_HORIZONTAL_LINEWIDTH, alpha=PCS_HORIZONTAL_ALPHA)

ax_seq.set_xlim(-WINDOW_HOURS, 0); ax_seq.set_yticks(clusters); ax_seq.set_ylabel("DEC state")
ax_seq.tick_params(labelbottom=False)
ax_seq.grid(True, color=GRID_COLOR, lw=SEQUENCE_GRID_LINEWIDTH, alpha=SEQUENCE_GRID_ALPHA)
ax_seq.text(
    *PANEL_LABEL_POSITION["a"], "a", transform=ax_seq.transAxes,
    fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top"
)

for i, cid in enumerate(clusters):
    ax = fig.add_subplot(left[i + 1, 0], sharex=ax_seq)
    hit = (labels == cid).astype(int); cum = np.cumsum(hit)
    n_state = int(hit.sum()); n_scale = max(n_state, 1)
    color = pcs_color if cid == pcs_cluster else cluster_colors[cid]
    line_alpha = CUMULATIVE_PCS_ALPHA if cid == pcs_cluster else CUMULATIVE_OTHER_ALPHA
    line_width = CUMULATIVE_PCS_LINEWIDTH if cid == pcs_cluster else CUMULATIVE_OTHER_LINEWIDTH

    for x in time_h[hit == 1]:
        ax.axvline(
            x, color=color, lw=OCCURRENCE_LINEWIDTH,
            alpha=OCCURRENCE_PCS_ALPHA if cid == pcs_cluster else OCCURRENCE_OTHER_ALPHA
        )

    ax.plot(time_h, cum, color=color, lw=line_width, alpha=line_alpha)
    ax.fill_between(
        time_h, 0, cum, color=color,
        alpha=CUMULATIVE_PCS_FILL_ALPHA if cid == pcs_cluster else CUMULATIVE_OTHER_FILL_ALPHA
    )
    ax.set_ylim(0, n_scale); ax.set_ylabel("Cum. count", labelpad=CUMULATIVE_COUNT_LABEL_PAD)
    ax.grid(True, color=GRID_COLOR, lw=CUMULATIVE_GRID_LINEWIDTH, alpha=CUMULATIVE_GRID_ALPHA)

    ax_prob = ax.secondary_yaxis(
        "right",
        functions=(
            lambda count, n=n_scale: count / n,
            lambda fraction, n=n_scale: fraction * n
        )
    )
    ax_prob.set_yticks([0.0, 0.5, 1.0]); ax_prob.set_yticklabels(["0", "0.5", "1"])
    ax_prob.tick_params(
        axis="y", labelsize=FONT_SIZES["tick"],
        pad=SECONDARY_AXIS_TICK_PAD, length=SECONDARY_AXIS_TICK_LENGTH
    )

    if i == n_clusters // 2:
        ax_prob.set_ylabel(
            "Probability", fontsize=FONT_SIZES["axis_label"],
            labelpad=CUMULATIVE_FRACTION_LABEL_PAD
        )

    ax.text(
        *STATE_LABEL_POSITION, f"C{cid}" + (" | PCS" if cid == pcs_cluster else ""),
        transform=ax.transAxes, color=pcs_color if cid == pcs_cluster else "#555555",
        fontsize=FONT_SIZES["annotation"],
        fontweight="bold" if cid == pcs_cluster else "normal"
    )

    if i < n_clusters - 1:
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel("Time before debris-flow initiation (h)")

    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color); spine.set_linewidth(PCS_SPINE_LINEWIDTH)


# b. PSD probability-density distributions
psd_grid = GridSpecFromSubplotSpec(1, n_clusters, subplot_spec=outer[0, 1], wspace=PSD_WSPACE)
psd_axes = []; pdf_mappable = None

for j, cid in enumerate(clusters):
    ax = fig.add_subplot(psd_grid[0, j]); psd_axes.append(ax)
    den = density_by_cluster[cid]

    pdf_mappable = ax.pcolormesh(
        freq_bins, psd_bins, den.T, cmap=PSD_CMAP, shading="auto",
        vmin=0, vmax=vmax_pdf, rasterized=True
    )

    median_psd = np.nanmedian(psd_by_cluster[cid], axis=0)
    ax.plot(
        freq_ref, median_psd,
        color=pcs_color if cid == pcs_cluster else NORMAL_COLOR,
        lw=PSD_MEDIAN_LINEWIDTH
    )

    ax.set_xlim(*PSD_FREQUENCY_LIMIT); ax.set_ylim(psd_min, psd_max)
    ax.set_title(
        f"C{cid}" + (" | PCS" if cid == pcs_cluster else "") + f"\nn={(labels == cid).sum()}",
        color=pcs_color if cid == pcs_cluster else "black",
        fontweight="bold" if cid == pcs_cluster else "normal",
        pad=PSD_TITLE_PAD
    )
    ax.set_xlabel("Frequency (Hz)")
    ax.grid(True, color=PSD_GRID_COLOR, lw=PSD_GRID_LINEWIDTH, alpha=PSD_GRID_ALPHA)

    if j == 0:
        ax.set_ylabel("PSD (dB)")
        ax.text(
            *PANEL_LABEL_POSITION["b"], "b", transform=ax.transAxes,
            fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top"
        )
    else:
        ax.tick_params(labelleft=False)

    if cid == pcs_cluster:
        for spine in ax.spines.values():
            spine.set_color(pcs_color); spine.set_linewidth(PSD_PCS_SPINE_LINEWIDTH)


# c. Representative waveform and STFT samples
stft_grid = GridSpecFromSubplotSpec(
    N_REPRESENTATIVES, n_clusters, subplot_spec=outer[1, 1],
    wspace=STFT_WSPACE, hspace=STFT_HSPACE
)
stft_axes = []; last_im = None

for j, cid in enumerate(clusters):
    reps = rep_by_cluster[cid][:N_REPRESENTATIVES]

    for r in range(N_REPRESENTATIVES):
        cell = GridSpecFromSubplotSpec(
            2, 1, subplot_spec=stft_grid[r, j],
            height_ratios=WAVEFORM_STFT_HEIGHT_RATIOS,
            hspace=WAVEFORM_STFT_HSPACE
        )
        ax_w = fig.add_subplot(cell[0, 0]); ax_s = fig.add_subplot(cell[1, 0])
        stft_axes.append(ax_s)

        if r < len(reps):
            x = waveform_for_index(reps[r]); t = np.arange(len(x)) / fs
            scale = np.nanpercentile(np.abs(x), WAVEFORM_NORMALIZATION_PERCENTILE)
            x_norm = x / scale if scale > 0 else x
            ax_w.plot(t, x_norm, color=WAVEFORM_COLOR, lw=WAVEFORM_LINEWIDTH)

            nperseg_use = min(STFT_NPERSEG, len(x))
            noverlap_use = min(STFT_NOVERLAP, max(0, nperseg_use - 1))
            f, tt, zz = stft(x, fs=fs, nperseg=nperseg_use, noverlap=noverlap_use)

            keep = f <= STFT_FREQUENCY_LIMIT[1]
            amp = np.abs(zz[keep])
            amp_scale = np.nanpercentile(amp, STFT_NORMALIZATION_PERCENTILE) + 1e-12
            amp = np.clip(amp / amp_scale, STFT_VMIN, STFT_VMAX)

            last_im = ax_s.pcolormesh(
                tt, f[keep], amp, cmap=STFT_CMAP, shading="auto",
                vmin=STFT_VMIN, vmax=STFT_VMAX, rasterized=True
            )

        ax_w.set_xlim(*WAVEFORM_TIME_LIMIT); ax_w.set_ylim(*WAVEFORM_AMPLITUDE_LIMIT)
        ax_s.set_xlim(*WAVEFORM_TIME_LIMIT); ax_s.set_ylim(*STFT_FREQUENCY_LIMIT)

        for ax in (ax_w, ax_s):
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_linewidth(NORMAL_SPINE_LINEWIDTH); spine.set_color(NORMAL_SPINE_COLOR)

        if cid == pcs_cluster:
            for ax in (ax_w, ax_s):
                for spine in ax.spines.values():
                    spine.set_color(pcs_color); spine.set_linewidth(PCS_STFT_SPINE_LINEWIDTH)

        if j == 0 and r == 0:
            ax_w.text(
                *PANEL_LABEL_POSITION["c"], "c", transform=ax_w.transAxes,
                fontsize=FONT_SIZES["panel_label"], fontweight="bold", va="top"
            )

        if r == N_REPRESENTATIVES - 1:
            ax_s.set_xticks([0, 30, 60])
            ax_s.set_xlabel("Time (s)", labelpad=BOTTOM_TIME_LABEL_PAD)

        if j == 0:
            ax_s.set_yticks([0, 25, 50])
            if r == N_REPRESENTATIVES // 2:
                ax_s.set_ylabel("Frequency (Hz)", labelpad=LEFT_FREQUENCY_LABEL_PAD)


# Independent colorbars
fig.canvas.draw()

bbox_psd = Bbox.union([ax.get_position() for ax in psd_axes])
cax_pdf = fig.add_axes([
    bbox_psd.x1 + PSD_CB_XPAD, bbox_psd.y0 + PSD_CB_YSHRINK,
    PSD_CB_WIDTH, bbox_psd.height - 2 * PSD_CB_YSHRINK
])
cb = fig.colorbar(pdf_mappable, cax=cax_pdf, orientation="vertical")
cb.set_label(
    "Probability density (%)",
    fontsize=FONT_SIZES["colorbar_label"],
    labelpad=COLORBAR_LABEL_PAD
)
cb.ax.tick_params(
    labelsize=FONT_SIZES["colorbar_tick"],
    pad=COLORBAR_TICK_PAD,
    length=COLORBAR_TICK_LENGTH
)

bbox_stft = Bbox.union([ax.get_position() for ax in stft_axes])
cax_stft = fig.add_axes([
    bbox_stft.x1 + STFT_CB_XPAD, bbox_stft.y0 + STFT_CB_YSHRINK,
    STFT_CB_WIDTH, bbox_stft.height - 2 * STFT_CB_YSHRINK
])
cb2 = fig.colorbar(last_im, cax=cax_stft, orientation="vertical")
cb2.set_label(
    "Normalized amplitude",
    fontsize=FONT_SIZES["colorbar_label"],
    labelpad=COLORBAR_LABEL_PAD
)
cb2.ax.tick_params(
    labelsize=FONT_SIZES["colorbar_tick"],
    pad=COLORBAR_TICK_PAD,
    length=COLORBAR_TICK_LENGTH
)


# 全框线，保留原边框颜色和线宽
for ax in fig.axes:
    if ax in (cax_pdf,cax_stft):
        continue
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(axis="both",which="both",direction="out",bottom=True,left=True,top=False,right=False,labeltop=False,labelright=False)

# 色条边框与刻度
for colorbar in (cb,cb2):
    colorbar.outline.set_visible(True)
    colorbar.ax.tick_params(direction="out")

# 保存 Fig 5
from pathlib import Path
save_root=Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure\Fig_all")
(save_root/"png").mkdir(parents=True,exist_ok=True)
(save_root/"tiff").mkdir(parents=True,exist_ok=True)
png_path=save_root/"png"/"Fig 5.png"
tiff_path=save_root/"tiff"/"Fig 5.tiff"
fig.savefig(png_path,format="png",dpi=SAVE_DPI,bbox_inches="tight",facecolor="white")
fig.savefig(tiff_path,format="tiff",dpi=600,bbox_inches="tight",facecolor="white",pil_kwargs={"compression":"tiff_lzw"})
plt.show()
plt.close(fig)
print("Saved Fig 5:")
print(png_path)
print(tiff_path)